# Assignment 3: Bus Mapping via Network Structure

This notebook maps each **Dayzer bus** to a likely **Panorama bus** using bus metadata and branch-network structure.

## My idea

The assignment asks for a bus-to-bus mapping, but I do not want to treat it as only a string-matching problem. In a power-grid model, two buses can have different names but still occupy the same electrical neighborhood. Also, two data sources may split or merge station-level buses differently.

My approach is therefore:

1. build raw bus-branch graphs for Dayzer and Panorama;
2. create a conservative **station-kV group proxy** to handle possible topology/modeling granularity differences;
3. match Dayzer groups to Panorama groups using name, voltage, geography, and local topology;
4. refine the group-level result back to the required raw-bus output columns: `dayzer_bus` and `pano_bus`;
5. keep ambiguous and group-level cases marked for review instead of forcing every row into a false one-to-one match.

## How this differs from a simpler raw bus-to-bus method

A simpler method would compare every Dayzer bus directly with Panorama buses using fuzzy names and coordinates. I think that can be misleading because the two model sources may not use the same bus splitting convention. This notebook first aligns station-kV group proxies and then produces a raw-bus representative result. Transformer high-side and low-side buses are **not** collapsed, because they are different voltage-level nodes in a power-flow model.


## Step 0 — Imports, configuration, and output paths

This cell imports the libraries, defines file paths, and sets thresholds/weights.

Why this step matters:

- Keeping thresholds at the top makes the method easier to review.
- The broad candidate search keeps more possible matches before final scoring.
- Runtime settings are configurable, but they do not change the conceptual mapping method.
- The notebook writes only the required submission files: one matched-result CSV and one summary report.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import math
import re
import json
import warnings
import os
import time

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from joblib import Parallel, delayed
    JOBLIB_AVAILABLE = True
except Exception:
    Parallel = None
    delayed = None
    JOBLIB_AVAILABLE = False

try:
    from rapidfuzz import fuzz, process
    RAPIDFUZZ_AVAILABLE = True
except Exception:
    from difflib import SequenceMatcher
    RAPIDFUZZ_AVAILABLE = False

# ------------------------------
# Input / output paths
# ------------------------------
# Keep paths relative so the submission folder can be moved or cloned elsewhere.
def find_submission_root() -> Path:
    candidates = [
        Path("."),
        Path(".."),
        Path("../.."),
        Path("ecesis-assignment-submission"),
        Path("../ecesis-assignment-submission"),
        Path("../../ecesis-assignment-submission"),
        Path("Summer2026-main/ecesis-assignment-submission"),
    ]
    for candidate in candidates:
        data_file = candidate / "data" / "Assignment 3 - bus_mapping" / "DAYZER_BUS_GEO.csv"
        notebook_dir = candidate / "assignment3_bus_mapping" / "notebook"
        if data_file.exists() and notebook_dir.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the submission root with data/Assignment 3 - bus_mapping. "
        "Run this notebook from the submission root, assignment3_bus_mapping/notebook, "
        "or the parent folder that contains Summer2026-main."
    )

PROJECT_ROOT = find_submission_root()
DATA_DIR = PROJECT_ROOT / "data" / "Assignment 3 - bus_mapping"
ASSIGNMENT_DIR = PROJECT_ROOT / "assignment3_bus_mapping"
OUT_DIR = ASSIGNMENT_DIR / "outputs"
REPORT_DIR = ASSIGNMENT_DIR / "report"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DAYZER_BUS_FILE = DATA_DIR / "DAYZER_BUS_GEO.csv"
DAYZER_BRANCH_FILE = DATA_DIR / "DAYZER_BRANCH.csv"
PANO_BUS_FILE = DATA_DIR / "PANO_BUS_GEO.csv"
PANO_BRANCH_FILE = DATA_DIR / "PANO_BRANCH.csv"

# ------------------------------
# Runtime settings
# ------------------------------
# These settings make the broad candidate search practical on a multi-core machine.
# They are not part of the matching definition itself.
CPU_COUNT = os.cpu_count() or 1
N_JOBS = min(16, max(1, CPU_COUNT - 2))
RAPIDFUZZ_WORKERS = N_JOBS if RAPIDFUZZ_AVAILABLE else 1
USE_PARALLEL_MATCHED_NEIGHBOR = JOBLIB_AVAILABLE and N_JOBS > 1
PARALLEL_BACKEND = "loky"
STRING_SCORE_CHUNK_SIZE = 250_000
PROFILE_SCORE_CHUNK_SIZE = 250_000
MATCHED_NEIGHBOR_CHUNK_SIZE = 150_000

# Lower N_JOBS if you want the computer to stay more responsive while running.

# ------------------------------
# Candidate-generation settings
# ------------------------------
# These settings deliberately keep a wider candidate set. This is slower,
# but it reduces the risk that the correct match is lost during early blocking.
TOP_FUZZY_CANDIDATES_PER_GROUP = 80
TOP_GEO_CANDIDATES_PER_GROUP = 30
TOP_DEGREE_CANDIDATES_PER_GROUP = 25
MAX_GROUP_CANDIDATES = 140
GEO_RECOVERY_MAX_KM = 150.0
EXPORT_TOP_CANDIDATES_PER_GROUP = 20

# If set to True, the notebook will attempt to include all same-kV group pairs.
# This is usually unnecessary and can create ~25 million candidate pairs on this
# dataset. Keep False for accuracy-first but tractable execution.
EXHAUSTIVE_SAME_KV_CANDIDATES = False

# ------------------------------
# Scoring thresholds
# ------------------------------
SAME_KV_TOLERANCE = 1.0
AMBIGUOUS_MARGIN = 0.05
HIGH_CONF_THRESHOLD = 0.85
MED_CONF_THRESHOLD = 0.70
LOW_CONF_THRESHOLD = 0.50

# ------------------------------
# Group-level score weights
# ------------------------------
ATTRIBUTE_WEIGHTS = {
    "station_score": 0.35,
    "name_score": 0.20,
    "voltage_score": 0.20,
    "geo_score": 0.25,
}

TOPOLOGY_WEIGHTS = {
    "degree_score": 0.25,
    "neighbor_kv_score": 0.25,
    "branch_type_score": 0.25,
    "matched_neighbor_score": 0.25,
}

FINAL_WEIGHTS = {
    "attribute_score": 0.60,
    "topology_score": 0.40,
}

print("RapidFuzz available:", RAPIDFUZZ_AVAILABLE)
print("Joblib available:", JOBLIB_AVAILABLE)
print("CPU count:", CPU_COUNT)
print("N_JOBS:", N_JOBS)
print("RapidFuzz workers:", RAPIDFUZZ_WORKERS)
print("Output directory:", OUT_DIR)



## Step 1 — Load and inspect the raw data

This cell loads the four input CSV files and prints basic schema information.

Why this step matters:

- Before matching, we need to understand the columns, missing data, and row counts.
- It also verifies that the branch files can be linked to the bus files through bus names.

In [2]:
dayzer_bus_raw = pd.read_csv(DAYZER_BUS_FILE)
dayzer_branch_raw = pd.read_csv(DAYZER_BRANCH_FILE)
pano_bus_raw = pd.read_csv(PANO_BUS_FILE)
pano_branch_raw = pd.read_csv(PANO_BRANCH_FILE)

print("Raw table shapes")
print("Dayzer bus:   ", dayzer_bus_raw.shape)
print("Dayzer branch:", dayzer_branch_raw.shape)
print("Panorama bus: ", pano_bus_raw.shape)
print("Panorama branch:", pano_branch_raw.shape)

for name, df in [
    ("DAYZER_BUS_GEO", dayzer_bus_raw),
    ("DAYZER_BRANCH", dayzer_branch_raw),
    ("PANO_BUS_GEO", pano_bus_raw),
    ("PANO_BRANCH", pano_branch_raw),
]:
    print("\n" + "=" * 80)
    print(name)
    print("Columns:", list(df.columns))
    print("Missing values:")
    display(df.isna().sum().to_frame("missing_count").T)
    display(df.head(3))

Raw table shapes
Dayzer bus:    (10799, 8)
Dayzer branch: (13621, 15)
Panorama bus:  (10233, 9)
Panorama branch: (11267, 16)

DAYZER_BUS_GEO
Columns: ['MDB_ID', 'NAME', 'BASKV', 'IDE', 'AREA_NAME', 'ZONE_NAME', 'LAT', 'LON']
Missing values:


,MDB_ID,NAME,BASKV,IDE,AREA_NAME,ZONE_NAME,LAT,LON
missing_count,0,0,0,0,0,0,681,681


,MDB_ID,NAME,BASKV,IDE,AREA_NAME,ZONE_NAME,LAT,LON
0,1,ROANSPRARE,138.0,1,BEPC_TSP,BEPC_GRIMES,30.584801,-95.934303
1,1,KEITHSW,138.0,1,BEPC_TSP,BEPC_GRIMES,30.650999,-96.089996
2,1,IOLA,138.0,1,BEPC_TSP,BEPC_GRIMES,30.730000,-96.044998



DAYZER_BRANCH
Columns: ['MDB_ID', 'NAME', 'BRANCH_TYPE', 'FROM_BUS', 'TO_BUS', 'FROM_KV', 'TO_KV', 'CKT', 'R', 'X', 'B', 'SRATEA', 'SRATEB', 'WRATEA', 'WRATEB']
Missing values:


,MDB_ID,NAME,BRANCH_TYPE,FROM_BUS,TO_BUS,FROM_KV,TO_KV,CKT,R,X,B,SRATEA,SRATEB,WRATEA,WRATEB
missing_count,0,17,0,0,0,0,0,0,0,0,2754,0,0,0,0


...

,NAME,BASKV,BUS_TYPE,AREA_NAME,ZONE_NAME,LAT,LON,PANO_ID,STATION
missing_count,0,0,0,0,0,0,0,0,10


,NAME,BASKV,BUS_TYPE,AREA_NAME,ZONE_NAME,LAT,LON,PANO_ID,STATION
0,ADK_138KV_1,138.0,LOAD,ERC,COAS,29.976728,-95.594192,1,ADK
1,ADK_138KV_2,138.0,LOAD,ERC,COAS,29.976539,-95.595559,1,ADK
2,ADK_138KV_3,138.0,ISOLATED,ERC,COAS,29.976264,-95.593649,1,ADK



PANO_BRANCH
Columns: ['NAME', 'BRANCH_TYPE', 'FROM_BUS_ID', 'FROM_BUS_KV', 'TO_BUS_ID', 'TO_BUS_KV', 'CIRCUIT', 'R', 'X', 'B', 'RATE_A', 'RATE_B', 'RATE_C', 'FROM_BUS_NAME', 'TO_BUS_NAME', 'PANO_ID']
Missing values:


,NAME,BRANCH_TYPE,FROM_BUS_ID,FROM_BUS_KV,TO_BUS_ID,TO_BUS_KV,CIRCUIT,R,X,B,RATE_A,RATE_B,RATE_C,FROM_BUS_NAME,TO_BUS_NAME,PANO_ID
missing_count,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


,NAME,BRANCH_TYPE,FROM_BUS_ID,FROM_BUS_KV,TO_BUS_ID,TO_BUS_KV,CIRCUIT,R,X,B,RATE_A,RATE_B,RATE_C,FROM_BUS_NAME,TO_BUS_NAME,PANO_ID
0,0165_1,LN,5024,345.0,5116,345.0,1,0.00046,0.00696,0.14096,1480.00,1483.00,1793.00,W_DENT_345KV_1,RNKSW_345KV_1,1
1,100016_1,LN,4942,138.0,6139,138.0,1,0.00001,0.00050,0.00000,281.29,281.29,300.11,LVOAK_138KV_1,LIVEOAK_138KV_1,1
2,100018_1_,LN,5880,69.0,5894,69.0,1,0.00001,0.00050,0.00000,87.00,87.00,100.85,ORANS_69KV_1,ORN_69KV_1,1


## Step 2 — Helper functions for normalization and scoring

This cell defines functions used throughout the notebook.

Main functions:

- `normalize_name`: cleans names into comparable uppercase tokens.
- `station_guess_from_dayzer`: derives a station-like name from a Dayzer bus name.
- `station_norm_from_pano`: uses Panorama's station field when available.
- `haversine_km`: computes geographic distance.
- `safe_similarity`: fuzzy string similarity.
- `profile_similarity`: compares graph-neighborhood profiles such as branch-type counts or neighbor-kV counts.

Why this step matters:

- Different sources use different naming conventions.
- We need robust, reusable functions so the matching logic is explainable.

In [3]:
def normalize_name(value):
    """Normalize a bus/station name for comparison.

    The goal is not to erase all information, but to make obvious formatting
    differences less important: spaces, punctuation, and case.
    """
    if pd.isna(value):
        return ""
    s = str(value).upper().strip()
    s = re.sub(r"[^A-Z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def remove_voltage_tokens(s):
    """Remove voltage-like tokens when comparing station names."""
    s = normalize_name(s)
    s = re.sub(r"_(?:\d+(?:\.\d+)?)KV(?=_|$)", "", s)
    s = re.sub(r"(?:^|_)(?:\d+(?:\.\d+)?)KV(?=_|$)", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def station_guess_from_dayzer(bus_name):
    """Derive a station-like key from a Dayzer bus name.

    Dayzer names often contain suffixes such as _5, _8, or _9. Those suffixes
    can represent voltage or modeling conventions rather than the station name.
    This function removes only simple trailing suffixes for station-level blocking.
    """
    s = remove_voltage_tokens(bus_name)
    # Remove simple trailing numeric suffixes after underscore.
    s = re.sub(r"_(?:\d+)$", "", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or normalize_name(bus_name)


def station_norm_from_pano(row):
    """Use Panorama's STATION column when available; otherwise derive from bus NAME."""
    station = row.get("STATION", "")
    if pd.notna(station) and str(station).strip():
        return normalize_name(station)
    name = normalize_name(row.get("NAME", ""))
    # Remove typical Panorama suffixes like _138KV_1.
    name = re.sub(r"_\d+(?:\.\d+)?KV(?:_\d+)?$", "", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def kv_key(kv):
    """Convert a voltage value into a rounded string key."""
    if pd.isna(kv):
        return "NA"
    try:
        return str(int(round(float(kv))))
    except Exception:
        return "NA"


def safe_float(x):
    """Return float(x), or NaN if conversion fails."""
    try:
        return float(x)
    except Exception:
        return np.nan


def safe_similarity(a, b):
    """Return fuzzy similarity in [0, 1]."""
    a = normalize_name(a)
    b = normalize_name(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    if RAPIDFUZZ_AVAILABLE:
        return fuzz.WRatio(a, b) / 100.0
    return SequenceMatcher(None, a, b).ratio()


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance between two latitude/longitude points."""
    vals = [lat1, lon1, lat2, lon2]
    if any(pd.isna(v) for v in vals):
        return np.nan
    lat1, lon1, lat2, lon2 = map(float, vals)
    r = 6371.0088
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def geo_score_from_distance(distance_km):
    """Convert distance into a bounded score.

    Missing coordinates get a neutral-low score instead of being discarded,
    because one source may have missing or noisy geocoding.
    """
    if pd.isna(distance_km):
        return 0.45
    if distance_km <= 2:
        return 1.00
    if distance_km <= 10:
        return 0.90
    if distance_km <= 30:
        return 0.70
    if distance_km <= 100:
        return 0.35
    return 0.05


def voltage_score(kv1, kv2, tolerance=SAME_KV_TOLERANCE):
    """Score voltage compatibility."""
    if pd.isna(kv1) or pd.isna(kv2):
        return 0.50
    diff = abs(float(kv1) - float(kv2))
    if diff <= tolerance:
        return 1.0
    return 0.0


def degree_similarity(d1, d2):
    """Similarity between two graph degrees."""
    d1 = 0 if pd.isna(d1) else int(d1)
    d2 = 0 if pd.isna(d2) else int(d2)
    if d1 == 0 and d2 == 0:
        return 1.0
    return 1.0 - abs(d1 - d2) / max(d1, d2, 1)


def profile_similarity(profile_a, profile_b):
    """Cosine/Jaccard-like similarity for Counter dictionaries.

    Used for comparing branch-type distributions and neighbor-kV distributions.
    """
    if not isinstance(profile_a, dict):
        profile_a = {}
    if not isinstance(profile_b, dict):
        profile_b = {}
    keys = set(profile_a) | set(profile_b)
    if not keys:
        return 1.0
    numerator = sum(min(profile_a.get(k, 0), profile_b.get(k, 0)) for k in keys)
    denominator = sum(max(profile_a.get(k, 0), profile_b.get(k, 0)) for k in keys)
    return numerator / denominator if denominator else 1.0


def score_status(final_score, margin):
    """Assign a human-readable confidence status."""
    if pd.isna(final_score) or final_score < LOW_CONF_THRESHOLD:
        return "unmatched"
    if margin < AMBIGUOUS_MARGIN:
        return "ambiguous"
    if final_score >= HIGH_CONF_THRESHOLD:
        return "matched_high"
    if final_score >= MED_CONF_THRESHOLD:
        return "matched_medium"
    return "matched_low"


def compact_counter(counter_obj):
    """Convert Counter to a normal dict with sorted keys for stable CSV output."""
    return dict(sorted(counter_obj.items(), key=lambda x: str(x[0])))

## Step 3 — Standardize bus tables and deduplicate Dayzer buses

This cell creates clean bus tables with consistent internal column names.

Why this step matters:

- The raw Dayzer bus table contains duplicate bus names.
- The branch file refers to buses by name, so duplicate names can create artificial conflicts.
- This step deduplicates Dayzer by bus name and keeps the row with better metadata, especially available coordinates.

Key output variables:

- `dayzer_bus`: cleaned Dayzer bus table, one row per Dayzer bus name.
- `pano_bus`: cleaned Panorama bus table.
- `dayzer_duplicate_report`: duplicated Dayzer bus-name records for review.

In [4]:
# ------------------------------
# Standardize Dayzer bus table
# ------------------------------
dayzer_bus = dayzer_bus_raw.copy()
dayzer_bus["source"] = "DAYZER"
dayzer_bus["bus_name"] = dayzer_bus["NAME"].astype(str)
dayzer_bus["kv"] = pd.to_numeric(dayzer_bus["BASKV"], errors="coerce")
dayzer_bus["lat"] = pd.to_numeric(dayzer_bus["LAT"], errors="coerce")
dayzer_bus["lon"] = pd.to_numeric(dayzer_bus["LON"], errors="coerce")
dayzer_bus["name_norm"] = dayzer_bus["bus_name"].map(normalize_name)
dayzer_bus["station_norm"] = dayzer_bus["bus_name"].map(station_guess_from_dayzer)
dayzer_bus["kv_key"] = dayzer_bus["kv"].map(kv_key)
dayzer_bus["has_geo"] = dayzer_bus["lat"].notna() & dayzer_bus["lon"].notna()

# Report duplicate Dayzer bus names before deduplication.
dup_mask = dayzer_bus.duplicated("bus_name", keep=False)
dayzer_duplicate_report = dayzer_bus.loc[dup_mask].sort_values(["bus_name", "has_geo"], ascending=[True, False]).copy()
# Deduplicate: prefer rows with coordinates and then keep the first stable record.
dayzer_bus = (
    dayzer_bus.sort_values(["bus_name", "has_geo"], ascending=[True, False])
    .drop_duplicates("bus_name", keep="first")
    .reset_index(drop=True)
)

# ------------------------------
# Standardize Panorama bus table
# ------------------------------
pano_bus = pano_bus_raw.copy()
pano_bus["source"] = "PANORAMA"
pano_bus["bus_name"] = pano_bus["NAME"].astype(str)
pano_bus["kv"] = pd.to_numeric(pano_bus["BASKV"], errors="coerce")
pano_bus["lat"] = pd.to_numeric(pano_bus["LAT"], errors="coerce")
pano_bus["lon"] = pd.to_numeric(pano_bus["LON"], errors="coerce")
pano_bus["name_norm"] = pano_bus["bus_name"].map(normalize_name)
pano_bus["station_norm"] = pano_bus.apply(station_norm_from_pano, axis=1)
pano_bus["kv_key"] = pano_bus["kv"].map(kv_key)
pano_bus["has_geo"] = pano_bus["lat"].notna() & pano_bus["lon"].notna()

print("Dayzer raw bus rows:", len(dayzer_bus_raw))
print("Dayzer unique bus rows after deduplication:", len(dayzer_bus))
print("Dayzer duplicate-name rows identified for review:", len(dayzer_duplicate_report))
print("Panorama bus rows:", len(pano_bus))
print("Panorama unique bus names:", pano_bus["bus_name"].nunique())

display(dayzer_bus.head(3))
display(pano_bus.head(3))

Dayzer raw bus rows: 10799
Dayzer unique bus rows after deduplication: 10558
Dayzer duplicate-name rows identified for review: 471
Panorama bus rows: 10233
Panorama unique bus names: 10233


,MDB_ID,NAME,BASKV,IDE,AREA_NAME,ZONE_NAME,LAT,LON,source,bus_name,kv,lat,lon,name_norm,station_norm,kv_key,has_geo
0,1,1,1.0,1,STEC_TSP,STEC,28.358900,-98.795097,DAYZER,1,1.0,28.358900,-98.795097,1,1,1,True
1,1,1000,1.0,2,LCRA TSC,FAYETTE,29.961300,-96.688400,DAYZER,1000,1.0,29.961300,-96.688400,1000,1000,1,True
2,1,111138,1.0,1,E_CNP_TSP,E_GALVESTON,29.505131,-94.981339,DAYZER,111138,1.0,29.505131,-94.981339,111138,111138,1,True


,NAME,BASKV,BUS_TYPE,AREA_NAME,ZONE_NAME,LAT,LON,PANO_ID,STATION,source,bus_name,kv,lat,lon,name_norm,station_norm,kv_key,has_geo
0,ADK_138KV_1,138.0,LOAD,ERC,COAS,29.976728,-95.594192,1,ADK,PANORAMA,ADK_138KV_1,138.0,29.976728,-95.594192,ADK_138KV_1,ADK,138,True
1,ADK_138KV_2,138.0,LOAD,ERC,COAS,29.976539,-95.595559,1,ADK,PANORAMA,ADK_138KV_2,138.0,29.976539,-95.595559,ADK_138KV_2,ADK,138,True
2,ADK_138KV_3,138.0,ISOLATED,ERC,COAS,29.976264,-95.593649,1,ADK,PANORAMA,ADK_138KV_3,138.0,29.976264,-95.593649,ADK_138KV_3,ADK,138,True


## Step 4 — Create station-kV / electrical-group proxy IDs

This cell creates group IDs by combining normalized station identity and voltage level.

Why this step matters:

- A strict raw-bus one-to-one assumption may be wrong if one model splits a station-kV bus into several pieces and the other model merges it.
- Grouping by station and voltage gives a practical upper-level object for alignment.
- This is only a **proxy**, because the files do not include breaker/switch states.

Key output variables:

- `dayzer_group_id`: station-kV group proxy for each Dayzer bus.
- `pano_group_id`: station-kV group proxy for each Panorama bus.

In [5]:
def make_group_id(prefix, station_norm, kv_key_value):
    station_norm = normalize_name(station_norm) or "UNKNOWN"
    kv_key_value = str(kv_key_value) if pd.notna(kv_key_value) else "NA"
    return f"{prefix}_{station_norm}_{kv_key_value}KV"

dayzer_bus["group_id"] = [
    make_group_id("DZG", st, kv) for st, kv in zip(dayzer_bus["station_norm"], dayzer_bus["kv_key"])
]
pano_bus["group_id"] = [
    make_group_id("PNG", st, kv) for st, kv in zip(pano_bus["station_norm"], pano_bus["kv_key"])
]

print("Dayzer station-kV groups:", dayzer_bus["group_id"].nunique())
print("Panorama station-kV groups:", pano_bus["group_id"].nunique())

print("\nLargest Dayzer groups:")
display(dayzer_bus.groupby("group_id").size().sort_values(ascending=False).head(10).to_frame("member_count"))

print("\nLargest Panorama groups:")
display(pano_bus.groupby("group_id").size().sort_values(ascending=False).head(10).to_frame("member_count"))

Dayzer station-kV groups: 10441
Panorama station-kV groups: 7303

Largest Dayzer groups:


,member_count
group_id,
DZG_SL_12KV,7
DZG_DOWGEN_14KV,6
DZG_SL_34KV,5
DZG_BOSQ_BSQSU_18KV,4
DZG_DECKE_DPGT_14KV,4
DZG_HUD_8_138KV,4
DZG_NA_12KV,4
DZG_OLIN_OLING_14KV,4
DZG_PHOEBE_34KV,4



Largest Panorama groups:


,member_count
group_id,
PNG_DOWGEN_14KV,19
PNG_DOWGEN_138KV,18
PNG_FORMOSA_14KV,16
PNG_THW_14KV,16
PNG_WCRSW_138KV,10
PNG_THW_1KV,10
PNG_TOPAZ_14KV,10
PNG_WAP_138KV,9
PNG_BOG_12KV,8


## Step 5 — Build raw bus-branch graphs

This cell builds one raw graph per source.

Why this step matters:

- The branch files describe how buses are connected.
- Raw graph topology captures electrical neighborhoods.
- We preserve branch type and parameters such as R/X when available.
- Transformer edges are kept as edges, but not collapsed into a single bus.

Key output variables:

- `G_dz_raw`: Dayzer raw bus graph.
- `G_pn_raw`: Panorama raw bus graph.

In [6]:
def build_raw_graph(branch_df, source):
    """Build a raw bus-level MultiGraph from a branch table."""
    G = nx.MultiGraph(source=source)
    if source == "DAYZER":
        from_col, to_col = "FROM_BUS", "TO_BUS"
        from_kv_col, to_kv_col = "FROM_KV", "TO_KV"
        branch_type_col = "BRANCH_TYPE"
        name_col = "NAME"
        rate_col = "SRATEA" if "SRATEA" in branch_df.columns else None
    else:
        from_col, to_col = "FROM_BUS_NAME", "TO_BUS_NAME"
        from_kv_col, to_kv_col = "FROM_BUS_KV", "TO_BUS_KV"
        branch_type_col = "BRANCH_TYPE"
        name_col = "NAME"
        rate_col = "RATE_A" if "RATE_A" in branch_df.columns else None

    for _, row in branch_df.iterrows():
        u = row.get(from_col)
        v = row.get(to_col)
        if pd.isna(u) or pd.isna(v):
            continue
        u = str(u)
        v = str(v)
        attrs = {
            "branch_name": row.get(name_col, ""),
            "branch_type": normalize_name(row.get(branch_type_col, "")),
            "from_kv": safe_float(row.get(from_kv_col, np.nan)),
            "to_kv": safe_float(row.get(to_kv_col, np.nan)),
            "r": safe_float(row.get("R", np.nan)),
            "x": safe_float(row.get("X", np.nan)),
            "rate": safe_float(row.get(rate_col, np.nan)) if rate_col else np.nan,
        }
        G.add_edge(u, v, **attrs)
    return G


G_dz_raw = build_raw_graph(dayzer_branch_raw, "DAYZER")
G_pn_raw = build_raw_graph(pano_branch_raw, "PANORAMA")

# Add isolated bus-table nodes that might not appear in branch files.
G_dz_raw.add_nodes_from(dayzer_bus["bus_name"])
G_pn_raw.add_nodes_from(pano_bus["bus_name"])

print("Dayzer raw graph nodes/edges:", G_dz_raw.number_of_nodes(), G_dz_raw.number_of_edges())
print("Panorama raw graph nodes/edges:", G_pn_raw.number_of_nodes(), G_pn_raw.number_of_edges())

print("\\nDayzer branch types:")
display(dayzer_branch_raw["BRANCH_TYPE"].value_counts(dropna=False).to_frame("count"))

print("\\nPanorama branch types:")
display(pano_branch_raw["BRANCH_TYPE"].value_counts(dropna=False).to_frame("count"))

Dayzer raw graph nodes/edges: 10558 13621
Panorama raw graph nodes/edges: 10233 11267
\nDayzer branch types:


,count
BRANCH_TYPE,
LN,10867
XF,2754


\nPanorama branch types:


,count
BRANCH_TYPE,
LN,8535
XF,2732


## Step 6 — Build group-level graphs

This cell collapses raw buses into station-kV group proxies and builds group-level graphs.

Why this step matters:

- It lets us compare topology at a level that is less sensitive to bus-section split/merge differences.
- Edges inside the same group are recorded as `internal_edges`.
- Edges between groups become external topology features.

Important:

- This collapse is by station and voltage, not by transformer.
- A transformer connecting 345 kV and 138 kV remains an external edge between two different groups.

In [7]:
# Raw bus name -> group ID maps
dz_bus_to_group = dict(zip(dayzer_bus["bus_name"], dayzer_bus["group_id"]))
pn_bus_to_group = dict(zip(pano_bus["bus_name"], pano_bus["group_id"]))

# Group ID -> representative metadata maps
dz_group_to_kv = dayzer_bus.groupby("group_id")["kv"].median().to_dict()
pn_group_to_kv = pano_bus.groupby("group_id")["kv"].median().to_dict()


def build_group_graph(G_raw, bus_to_group, source):
    """Collapse a raw bus graph into a group-level graph."""
    G_group = nx.MultiGraph(source=source)
    internal_edges = Counter()
    internal_branch_types = defaultdict(Counter)

    for raw_node in G_raw.nodes:
        gid = bus_to_group.get(raw_node)
        if gid is not None:
            G_group.add_node(gid)

    for u, v, key, attrs in G_raw.edges(keys=True, data=True):
        gu = bus_to_group.get(u)
        gv = bus_to_group.get(v)
        if gu is None or gv is None:
            continue
        btype = normalize_name(attrs.get("branch_type", ""))
        if gu == gv:
            internal_edges[gu] += 1
            internal_branch_types[gu][btype] += 1
        else:
            G_group.add_edge(gu, gv, **attrs)

    return G_group, internal_edges, internal_branch_types


G_dz_group, dz_internal_edges, dz_internal_branch_types = build_group_graph(G_dz_raw, dz_bus_to_group, "DAYZER")
G_pn_group, pn_internal_edges, pn_internal_branch_types = build_group_graph(G_pn_raw, pn_bus_to_group, "PANORAMA")

print("Dayzer group graph nodes/edges:", G_dz_group.number_of_nodes(), G_dz_group.number_of_edges())
print("Panorama group graph nodes/edges:", G_pn_group.number_of_nodes(), G_pn_group.number_of_edges())

print("\nExample Dayzer groups with internal edges:")
display(pd.DataFrame(dz_internal_edges.items(), columns=["group_id", "internal_edge_count"]).sort_values("internal_edge_count", ascending=False).head(10))

print("\nExample Panorama groups with internal edges:")
display(pd.DataFrame(pn_internal_edges.items(), columns=["group_id", "internal_edge_count"]).sort_values("internal_edge_count", ascending=False).head(10))

Dayzer group graph nodes/edges: 10441 13397
Panorama group graph nodes/edges: 7303 10747

Example Dayzer groups with internal edges:


,group_id,internal_edge_count
62,DZG_SL_34KV,5
116,DZG_STONERSY_138KV,4
177,DZG_WATMLL_E5_345KV,4
101,DZG_MSN_34KV,3
40,DZG_TYLERGND_138KV,3
63,DZG_SL_12KV,3
125,DZG_NA_12KV,3
47,DZG_WACOESUB1_138KV,2
75,DZG_LEWISVLSW5_345KV,2
22,DZG_WATMLL_W5_345KV,2



Example Panorama groups with internal edges:


,group_id,internal_edge_count
171,PNG_DOWGEN_138KV,22
62,PNG_AMOCOOIL_138KV,8
96,PNG_WAP_138KV,8
218,PNG_NEDIN_138KV,7
172,PNG_DOWGEN_14KV,7
134,PNG_FORMOSA_138KV,7
102,PNG_SL_13KV,6
203,PNG_THW_138KV,6
188,PNG_EXN_69KV,5
65,PNG_HOC_138KV,5


## Step 7 — Compute raw-bus and group-level topology features

This cell computes graph features used for matching.

Raw-bus features:

- degree
- branch-type profile

Group-level features:

- member count
- centroid latitude/longitude
- external degree
- internal edge count
- branch-type profile
- neighbor-KV profile

Why this step matters:

- Two buses can have different names but similar network neighborhoods.
- Group-level topology is more robust when one dataset splits or merges bus sections differently.

In [8]:
def raw_node_features(G):
    """Compute simple topology features for each raw bus node."""
    rows = []
    for n in G.nodes:
        branch_types = Counter()
        neighbor_names = []
        for _, nbr, attrs in G.edges(n, data=True):
            branch_types[normalize_name(attrs.get("branch_type", ""))] += 1
            neighbor_names.append(nbr)
        rows.append({
            "bus_name": n,
            "raw_degree": G.degree(n),
            "raw_branch_type_profile": compact_counter(branch_types),
            "raw_neighbors": sorted(set(neighbor_names)),
        })
    return pd.DataFrame(rows)


dz_raw_feat = raw_node_features(G_dz_raw)
pn_raw_feat = raw_node_features(G_pn_raw)


def group_base_table(bus_df, source_prefix):
    """Aggregate bus metadata into station-kV group records."""
    rows = []
    for gid, sub in bus_df.groupby("group_id"):
        station = sub["station_norm"].mode().iloc[0] if len(sub["station_norm"].dropna()) else ""
        kv = sub["kv"].median()
        rows.append({
            "group_id": gid,
            "source": source_prefix,
            "station_norm": station,
            "kv": kv,
            "kv_key": kv_key(kv),
            "member_count": len(sub),
            "member_buses": sorted(sub["bus_name"].astype(str).tolist()),
            "name_corpus": " ".join(sorted(sub["name_norm"].astype(str).unique())),
            "lat": sub["lat"].mean(skipna=True),
            "lon": sub["lon"].mean(skipna=True),
            "has_geo": sub["lat"].notna().any() and sub["lon"].notna().any(),
        })
    return pd.DataFrame(rows)


dayzer_group = group_base_table(dayzer_bus, "DAYZER")
pano_group = group_base_table(pano_bus, "PANORAMA")


def add_group_topology_features(group_df, G_group, internal_edges, internal_branch_types, group_to_kv):
    """Add group-level graph features to a group table."""
    rows = []
    for _, row in group_df.iterrows():
        gid = row["group_id"]
        branch_types = Counter()
        neighbor_kvs = Counter()
        neighbor_groups = []

        if gid in G_group:
            for _, nbr, attrs in G_group.edges(gid, data=True):
                branch_types[normalize_name(attrs.get("branch_type", ""))] += 1
                neighbor_groups.append(nbr)
                neighbor_kvs[kv_key(group_to_kv.get(nbr, np.nan))] += 1

        new_row = row.to_dict()
        new_row.update({
            "external_degree": G_group.degree(gid) if gid in G_group else 0,
            "neighbor_group_count": len(set(neighbor_groups)),
            "neighbor_groups": sorted(set(neighbor_groups)),
            "neighbor_kv_profile": compact_counter(neighbor_kvs),
            "branch_type_profile": compact_counter(branch_types),
            "internal_edge_count": internal_edges.get(gid, 0),
            "internal_branch_type_profile": compact_counter(internal_branch_types.get(gid, Counter())),
        })
        rows.append(new_row)
    return pd.DataFrame(rows)


dayzer_group = add_group_topology_features(dayzer_group, G_dz_group, dz_internal_edges, dz_internal_branch_types, dz_group_to_kv)
pano_group = add_group_topology_features(pano_group, G_pn_group, pn_internal_edges, pn_internal_branch_types, pn_group_to_kv)

print("Dayzer group feature table:", dayzer_group.shape)
print("Panorama group feature table:", pano_group.shape)
display(dayzer_group.head(3))
display(pano_group.head(3))

Dayzer group feature table: (10441, 18)
Panorama group feature table: (7303, 18)


,group_id,source,station_norm,kv,kv_key,member_count,member_buses,name_corpus,lat,lon,has_geo,external_degree,neighbor_group_count,neighbor_groups,neighbor_kv_profile,branch_type_profile,internal_edge_count,internal_branch_type_profile
0,DZG_1000_1KV,DAYZER,1000,1.0,1,1,[1000],1000,29.961300,-96.688400,True,3,3,"[DZG_L_FAYETT1_2Y_13KV, DZG_L_FAYETT5_1Y_345KV...","{'13': 1, '138': 1, '345': 1}",{'XF': 3},0,{}
1,DZG_111138_1KV,DAYZER,111138,1.0,1,1,[111138],111138,29.505131,-94.981339,True,3,3,"[DZG_BAC_138KV, DZG_BAC_CTG1_14KV, DZG_BAC_CTG...","{'138': 1, '14': 2}",{'XF': 3},0,{}
2,DZG_111139_STAR_1KV,DAYZER,111139_STAR,1.0,1,1,[111139_STAR],111139_STAR,29.505131,-94.981339,True,3,3,"[DZG_BAC_138KV, DZG_BAC_CTG3_14KV, DZG_BAC_CTG...","{'138': 1, '14': 2}",{'XF': 3},0,{}


,group_id,source,station_norm,kv,kv_key,member_count,member_buses,name_corpus,lat,lon,has_geo,external_degree,neighbor_group_count,neighbor_groups,neighbor_kv_profile,branch_type_profile,internal_edge_count,internal_branch_type_profile
0,PNG_16TH_ST_138KV,PANORAMA,16TH_ST,138.0,138,1,[16TH_ST_138KV_1],16TH_ST_138KV_1,30.902090,-102.918367,True,2,2,"[PNG_16TH_ST_69KV, PNG_ALAMO_ST_138KV]","{'138': 1, '69': 1}","{'LN': 1, 'XF': 1}",0,{}
1,PNG_16TH_ST_69KV,PANORAMA,16TH_ST,69.0,69,1,[16TH_ST_69KV_1],16TH_ST_69KV_1,30.901960,-102.918745,True,2,2,"[PNG_16TH_ST_138KV, PNG_TNFS_69KV]","{'138': 1, '69': 1}","{'LN': 1, 'XF': 1}",0,{}
2,PNG_36POD_138KV,PANORAMA,36POD,138.0,138,1,[36POD_138KV_1],36POD_138KV_1,31.912397,-101.663448,True,2,2,"[PNG_DWIND_138KV, PNG_PDPOD_138KV]",{'138': 2},{'LN': 2},0,{}


## Step 8 — Generate group-level candidates with broad retrieval

This cell generates candidate Panorama groups for each Dayzer group.

Candidate logic:

1. Use voltage as a hard filter. A 138 kV group should not directly match a 345 kV group.
2. Keep exact station and prefix candidates.
3. Use RapidFuzz to retrieve many top station-name candidates inside each voltage bucket.
4. Add geographically nearest same-voltage candidates.
5. Add degree/topology-similar same-voltage candidates.
6. Use a relatively large candidate cap per Dayzer group.

Why this step matters:

- Precision can be lost if the first blocking stage is too narrow.
- This step is intentionally broad because it keeps more plausible candidates for later scoring.
- It still avoids blindly trusting names: the final decision uses voltage, geography, topology profiles, and matched-neighbor consistency.


Runtime note:

- The broad fuzzy retrieval uses `rapidfuzz.process.cdist(..., workers=RAPIDFUZZ_WORKERS)`.
- This is the first expensive part of the broad-candidate notebook, especially for the large 138 kV bucket.


In [9]:
# Prepare Panorama candidate pools by voltage key.
# Accuracy-first retrieval with route-priority candidate bookkeeping:
# - Hard filter by rounded kV.
# - Keep exact/prefix station candidates.
# - Use RapidFuzz cdist to retrieve broad same-kV fuzzy candidates.
# - Add nearest geographic candidates within the same kV bucket.
# - Add external-degree/topology-similar candidates within the same kV bucket.
# - Candidate priorities are recorded during retrieval, so we do not run expensive
#   fuzzy scoring over a huge union just to cap the list.

t0_step8 = time.perf_counter()

pano_by_kv = {
    kv: sub.reset_index(drop=True).copy()
    for kv, sub in pano_group.groupby("kv_key", dropna=False)
}

dayzer_by_kv = {
    kv: sub.reset_index(drop=True).copy()
    for kv, sub in dayzer_group.groupby("kv_key", dropna=False)
}


def compact_station_key(s):
    """Compact station key for exact/prefix retrieval."""
    return re.sub(r"[^A-Z0-9]", "", normalize_name(s))


def vectorized_haversine_km(lat1, lon1, lat2_array, lon2_array):
    """Vectorized distance from one point to arrays of candidate points."""
    if pd.isna(lat1) or pd.isna(lon1):
        return np.full(len(lat2_array), np.nan, dtype=float)
    lat2 = np.asarray(lat2_array, dtype=float)
    lon2 = np.asarray(lon2_array, dtype=float)
    valid = ~np.isnan(lat2) & ~np.isnan(lon2)
    out = np.full(len(lat2), np.nan, dtype=float)
    if not valid.any():
        return out
    r = 6371.0088
    phi1 = math.radians(float(lat1))
    lam1 = math.radians(float(lon1))
    phi2 = np.radians(lat2[valid])
    lam2 = np.radians(lon2[valid])
    dphi = phi2 - phi1
    dlam = lam2 - lam1
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2) ** 2
    out[valid] = 2 * r * np.arcsin(np.sqrt(a))
    return out


# Candidate score dictionary: candidate_priority[dayzer_gid][pano_gid] = best retrieval score.
# This score is used only to keep a broad but finite candidate list. Final scoring happens later.
candidate_priority = defaultdict(dict)


def add_candidate(dayzer_gid, pano_gid, score):
    """Add or update one candidate with the best retrieval-priority score."""
    old = candidate_priority[dayzer_gid].get(pano_gid, -1.0)
    if score > old:
        candidate_priority[dayzer_gid][pano_gid] = float(score)


def add_positions_with_score(dayzer_gid, pano_pool, positions, score):
    """Add positions from a Panorama pool using one shared retrieval score."""
    for pos in positions:
        if 0 <= int(pos) < len(pano_pool):
            add_candidate(dayzer_gid, pano_pool.loc[int(pos), "group_id"], score)


# Add helper columns and block dictionaries to Panorama pools.
block_maps = {}
for kv, sub in pano_by_kv.items():
    sub["station_compact"] = sub["station_norm"].map(compact_station_key)
    sub["prefix5"] = sub["station_compact"].str[:5]
    sub["prefix4"] = sub["station_compact"].str[:4]
    sub["prefix3"] = sub["station_compact"].str[:3]
    sub["first_token"] = sub["station_norm"].str.split("_").str[0].fillna("")
    sub["row_pos"] = np.arange(len(sub))
    pano_by_kv[kv] = sub

    maps = {
        "exact": defaultdict(list),
        "prefix5": defaultdict(list),
        "prefix4": defaultdict(list),
        "prefix3": defaultdict(list),
        "first_token": defaultdict(list),
    }
    for pos, row in sub.iterrows():
        maps["exact"][row["station_norm"]].append(pos)
        maps["prefix5"][row["prefix5"]].append(pos)
        maps["prefix4"][row["prefix4"]].append(pos)
        maps["prefix3"][row["prefix3"]].append(pos)
        maps["first_token"][row["first_token"]].append(pos)
    block_maps[kv] = maps


# Route 1: exact and prefix candidates within same kV.
for _, drow in dayzer_group.iterrows():
    kv = drow["kv_key"]
    if kv not in pano_by_kv:
        continue
    pano_pool = pano_by_kv[kv]
    maps = block_maps[kv]
    station = normalize_name(drow["station_norm"])
    compact = compact_station_key(station)
    first_token = station.split("_")[0] if station else ""

    add_positions_with_score(drow["group_id"], pano_pool, maps["exact"].get(station, []), 1.00)
    add_positions_with_score(drow["group_id"], pano_pool, maps["prefix5"].get(compact[:5], []), 0.88)
    add_positions_with_score(drow["group_id"], pano_pool, maps["prefix4"].get(compact[:4], []), 0.78)
    add_positions_with_score(drow["group_id"], pano_pool, maps["prefix3"].get(compact[:3], []), 0.68)
    add_positions_with_score(drow["group_id"], pano_pool, maps["first_token"].get(first_token, []), 0.72)

print("After exact/prefix retrieval, candidate pairs:", sum(len(v) for v in candidate_priority.values()))


# Route 2: broad fuzzy retrieval inside each same-kV bucket.
# For retrieval we use fuzz.ratio because it is much faster than WRatio on large same-kV buckets.
# Final scoring still uses safe_similarity, which uses WRatio when RapidFuzz is available.
if RAPIDFUZZ_AVAILABLE:
    for kv, dsub in dayzer_by_kv.items():
        if kv not in pano_by_kv:
            continue
        psub = pano_by_kv[kv]
        if dsub.empty or psub.empty:
            continue

        d_names = dsub["station_norm"].fillna("").astype(str).tolist()
        p_names = psub["station_norm"].fillna("").astype(str).tolist()
        k = min(TOP_FUZZY_CANDIDATES_PER_GROUP, len(psub))

        sim_matrix = process.cdist(
            d_names,
            p_names,
            scorer=fuzz.ratio,
            dtype=np.float32,
            workers=RAPIDFUZZ_WORKERS,
        )
        if k >= len(psub):
            top_idx = np.tile(np.arange(len(psub)), (len(dsub), 1))
            top_scores = sim_matrix
        else:
            part = np.argpartition(-sim_matrix, kth=k-1, axis=1)[:, :k]
            row_scores = np.take_along_axis(sim_matrix, part, axis=1)
            order = np.argsort(-row_scores, axis=1)
            top_idx = np.take_along_axis(part, order, axis=1)
            top_scores = np.take_along_axis(row_scores, order, axis=1)

        for row_i, dz_gid in enumerate(dsub["group_id"].values):
            for pos, score in zip(top_idx[row_i], top_scores[row_i]):
                add_candidate(dz_gid, psub.loc[int(pos), "group_id"], 0.15 + 0.85 * float(score) / 100.0)
        print(f"Fuzzy retrieval for kV={kv}: Dayzer groups={len(dsub):,}, Panorama groups={len(psub):,}, top-k={k}")
else:
    for kv, dsub in dayzer_by_kv.items():
        if kv not in pano_by_kv:
            continue
        psub = pano_by_kv[kv]
        for _, drow in dsub.iterrows():
            scored = [(pos, safe_similarity(drow["station_norm"], prow["station_norm"])) for pos, prow in psub.iterrows()]
            for pos, score in sorted(scored, key=lambda x: x[1], reverse=True)[:TOP_FUZZY_CANDIDATES_PER_GROUP]:
                add_candidate(drow["group_id"], psub.loc[int(pos), "group_id"], 0.15 + 0.85 * score)

print("After fuzzy retrieval, candidate pairs:", sum(len(v) for v in candidate_priority.values()))


# Route 3: geographic nearest candidates within the same kV bucket.
for kv, dsub in dayzer_by_kv.items():
    if kv not in pano_by_kv:
        continue
    psub = pano_by_kv[kv]
    if dsub.empty or psub.empty:
        continue
    plat = psub["lat"].to_numpy(dtype=float)
    plon = psub["lon"].to_numpy(dtype=float)
    k = min(TOP_GEO_CANDIDATES_PER_GROUP, len(psub))

    for _, drow in dsub.iterrows():
        dist = vectorized_haversine_km(drow["lat"], drow["lon"], plat, plon)
        finite = np.where(np.isfinite(dist))[0]
        if len(finite) == 0:
            continue
        within = finite[dist[finite] <= GEO_RECOVERY_MAX_KM]
        if len(within) > 0:
            sorted_pos = within[np.argsort(dist[within])[:k]]
        else:
            sorted_pos = finite[np.argsort(dist[finite])[:k]]
        for pos in sorted_pos:
            add_candidate(drow["group_id"], psub.loc[int(pos), "group_id"], 0.10 + 0.90 * geo_score_from_distance(dist[int(pos)]))

print("After geographic retrieval, candidate pairs:", sum(len(v) for v in candidate_priority.values()))


# Route 4: external-degree similar candidates within the same kV bucket.
for kv, dsub in dayzer_by_kv.items():
    if kv not in pano_by_kv:
        continue
    psub = pano_by_kv[kv]
    if dsub.empty or psub.empty:
        continue
    pdeg = psub["external_degree"].fillna(0).to_numpy(dtype=float)
    k = min(TOP_DEGREE_CANDIDATES_PER_GROUP, len(psub))

    for _, drow in dsub.iterrows():
        ddeg = 0.0 if pd.isna(drow["external_degree"]) else float(drow["external_degree"])
        degree_gap = np.abs(pdeg - ddeg)
        if k >= len(psub):
            top_pos = np.arange(len(psub))
        else:
            top_pos = np.argpartition(degree_gap, kth=k-1)[:k]
            top_pos = top_pos[np.argsort(degree_gap[top_pos])]
        for pos in top_pos:
            add_candidate(drow["group_id"], psub.loc[int(pos), "group_id"], 0.25 + 0.75 * degree_similarity(ddeg, pdeg[int(pos)]))

print("After degree/topology retrieval, candidate pairs:", sum(len(v) for v in candidate_priority.values()))


# Optional exhaustive mode: include every same-kV pair.
if EXHAUSTIVE_SAME_KV_CANDIDATES:
    for kv, dsub in dayzer_by_kv.items():
        if kv not in pano_by_kv:
            continue
        psub = pano_by_kv[kv]
        all_pano_gids = set(psub["group_id"].tolist())
        for dz_gid in dsub["group_id"]:
            for pgid in all_pano_gids:
                add_candidate(dz_gid, pgid, 0.50)
    print("After exhaustive same-kV retrieval, candidate pairs:", sum(len(v) for v in candidate_priority.values()))


# Cap candidates per Dayzer group after all retrieval routes have contributed.
candidate_records = []
for i, (dz_gid, pgid_scores) in enumerate(candidate_priority.items(), start=1):
    scored = sorted(pgid_scores.items(), key=lambda x: x[1], reverse=True)[:MAX_GROUP_CANDIDATES]
    for pgid, retrieval_score in scored:
        candidate_records.append({
            "dayzer_group_id": dz_gid,
            "pano_group_id": pgid,
            "retrieval_score": retrieval_score,
        })
    if i % 2000 == 0:
        print(f"Finalized candidates for {i:,} / {len(candidate_priority):,} Dayzer groups")


group_candidates = pd.DataFrame(candidate_records).drop_duplicates(["dayzer_group_id", "pano_group_id"]).reset_index(drop=True)

print("Generated group candidate pairs:", len(group_candidates))
print("Average candidates per Dayzer group:", len(group_candidates) / max(1, len(dayzer_group)))
print("Dayzer groups with at least one candidate:", group_candidates["dayzer_group_id"].nunique())
display(group_candidates.head())


print(f"Step 8 elapsed time: {time.perf_counter() - t0_step8:.1f} seconds")


After exact/prefix retrieval, candidate pairs: 26712
After fuzzy retrieval, candidate pairs: 816300
After geographic retrieval, candidate pairs: 1060596
After degree/topology retrieval, candidate pairs: 1280291
Finalized candidates for 2,000 / 10,441 Dayzer groups
Finalized candidates for 4,000 / 10,441 Dayzer groups
Finalized candidates for 6,000 / 10,441 Dayzer groups
Finalized candidates for 8,000 / 10,441 Dayzer groups
Finalized candidates for 10,000 / 10,441 Dayzer groups
Generated group candidate pairs: 1280213
Average candidates per Dayzer group: 122.61402164543627
Dayzer groups with at least one candidate: 10441


,dayzer_group_id,pano_group_id,retrieval_score
0,DZG_7RNCHSLR_138KV,PNG_7RNCHSLR_138KV,1.0
1,DZG_7RNCHSLR_138KV,PNG_CRSPOLRC_138KV,1.0
2,DZG_7RNCHSLR_138KV,PNG_CROSSETT_138KV,1.0
3,DZG_7RNCHSLR_138KV,PNG_CSETP_138KV,1.0
4,DZG_7RNCHSLR_138KV,PNG_CSEC_138KV,1.0


Step 8 elapsed time: 1086.1 seconds


## Step 9 — Score group-level candidates in chunked

This cell calculates full attribute and topology scores for each retained Dayzer-group / Panorama-group pair.

Attribute signals:

- station/name similarity
- voltage compatibility
- geographic distance

Topology signals:

- external degree similarity
- neighbor-kV profile similarity
- branch-type profile similarity

Why this step is slow:

- The broad-candidate version may keep more than one million candidate pairs.
- Each pair needs two fuzzy string scores and several topology-profile scores.
- This runtime setting uses RapidFuzz `cpdist(..., workers=RAPIDFUZZ_WORKERS)` and chunked array operations to reduce wall-clock time and memory spikes.

Scoring formula:

$$
S_{attr} = w_s S_{station} + w_n S_{name} + w_v S_{voltage} + w_g S_{geo}
$$

$$
S_{topo,pre} = 0.34S_{degree} + 0.33S_{neighborKV} + 0.33S_{branchType}
$$

$$
S_{pre} = 0.6S_{attr} + 0.4S_{topo,pre}
$$


In [10]:
t0_step9 = time.perf_counter()

# Score group-level candidates without a heavy object-column merge.
# The candidate set is broader than the speed-oriented notebook, so this cell uses
# array indexing and vectorized scoring wherever possible.

dz_table = dayzer_group.reset_index(drop=True).copy()
pn_table = pano_group.reset_index(drop=True).copy()

dz_gid_to_idx = {gid: i for i, gid in enumerate(dz_table["group_id"])}
pn_gid_to_idx = {gid: i for i, gid in enumerate(pn_table["group_id"])}

dz_idx = group_candidates["dayzer_group_id"].map(dz_gid_to_idx).to_numpy(dtype=int)
pn_idx = group_candidates["pano_group_id"].map(pn_gid_to_idx).to_numpy(dtype=int)


def fast_similarity(a, b):
    """Single-pair normalized string similarity for small fallback uses."""
    a = normalize_name(a)
    b = normalize_name(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    if RAPIDFUZZ_AVAILABLE:
        return fuzz.WRatio(a, b) / 100.0
    return SequenceMatcher(None, a, b).ratio()


def batch_similarity(left, right, chunk_size=STRING_SCORE_CHUNK_SIZE):
    """Pairwise string similarity for aligned candidate arrays.

    The work is split into chunks to avoid large temporary allocations. RapidFuzz
    is allowed to use multiple workers inside each chunk.
    """
    left = [normalize_name(x) for x in left]
    right = [normalize_name(x) for x in right]
    n = len(left)
    out = np.empty(n, dtype=np.float32)
    if RAPIDFUZZ_AVAILABLE:
        for start in range(0, n, chunk_size):
            end = min(start + chunk_size, n)
            out[start:end] = process.cpdist(
                left[start:end],
                right[start:end],
                scorer=fuzz.ratio,
                dtype=np.float32,
                workers=RAPIDFUZZ_WORKERS,
            ) / 100.0
            if start == 0 or end == n or (start // chunk_size) % 5 == 0:
                print(f"  string similarity scored {end:,} / {n:,} pairs")
        return out
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        out[start:end] = np.array(
            [SequenceMatcher(None, a, b).ratio() for a, b in zip(left[start:end], right[start:end])],
            dtype=np.float32,
        )
    return out


def haversine_numpy(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance for aligned numpy arrays."""
    lat1 = np.asarray(lat1, dtype=float)
    lon1 = np.asarray(lon1, dtype=float)
    lat2 = np.asarray(lat2, dtype=float)
    lon2 = np.asarray(lon2, dtype=float)
    valid = ~np.isnan(lat1) & ~np.isnan(lon1) & ~np.isnan(lat2) & ~np.isnan(lon2)
    out = np.full(len(lat1), np.nan, dtype=np.float32)
    if not valid.any():
        return out
    r = 6371.0088
    phi1 = np.radians(lat1[valid])
    phi2 = np.radians(lat2[valid])
    dphi = np.radians(lat2[valid] - lat1[valid])
    dlambda = np.radians(lon2[valid] - lon1[valid])
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    out[valid] = 2 * r * np.arcsin(np.sqrt(a))
    return out


def profile_keys(*series_list):
    """Collect sorted keys from dictionary-valued profile columns."""
    keys = set()
    for series in series_list:
        for obj in series:
            if isinstance(obj, dict):
                keys.update(obj.keys())
    return sorted(keys, key=lambda x: str(x))


def profile_matrix(df, profile_col, keys):
    """Convert a dictionary-valued profile column into a dense numeric matrix."""
    key_to_j = {k: j for j, k in enumerate(keys)}
    mat = np.zeros((len(df), len(keys)), dtype=np.float32)
    for i, prof in enumerate(df[profile_col]):
        if isinstance(prof, dict):
            for k, v in prof.items():
                j = key_to_j.get(k)
                if j is not None:
                    mat[i, j] = float(v)
    return mat


def profile_similarity_indexed(mat_a, idx_a, mat_b, idx_b, chunk_size=PROFILE_SCORE_CHUNK_SIZE):
    """Vectorized profile similarity for candidate index arrays."""
    out = np.empty(len(idx_a), dtype=np.float32)
    for start in range(0, len(idx_a), chunk_size):
        end = min(start + chunk_size, len(idx_a))
        a = mat_a[idx_a[start:end]]
        b = mat_b[idx_b[start:end]]
        num = np.minimum(a, b).sum(axis=1)
        den = np.maximum(a, b).sum(axis=1)
        out[start:end] = np.where(den == 0, 1.0, num / den)
    return out


def geo_score_array(distance):
    """Vectorized version of geo_score_from_distance."""
    d = np.asarray(distance, dtype=float)
    out = np.full(len(d), 0.45, dtype=np.float32)
    finite = np.isfinite(d)
    out[finite & (d <= 2)] = 1.00
    out[finite & (d > 2) & (d <= 10)] = 0.90
    out[finite & (d > 10) & (d <= 30)] = 0.70
    out[finite & (d > 30) & (d <= 100)] = 0.35
    out[finite & (d > 100)] = 0.05
    return out


# Pull candidate arrays by index.
dz_station = dz_table["station_norm"].to_numpy(dtype=object)[dz_idx]
pn_station = pn_table["station_norm"].to_numpy(dtype=object)[pn_idx]
dz_corpus = dz_table["name_corpus"].astype(str).str[:80].to_numpy(dtype=object)[dz_idx]
pn_corpus = pn_table["name_corpus"].astype(str).str[:80].to_numpy(dtype=object)[pn_idx]

dz_kv = dz_table["kv"].to_numpy(dtype=float)[dz_idx]
pn_kv = pn_table["kv"].to_numpy(dtype=float)[pn_idx]
dz_lat = dz_table["lat"].to_numpy(dtype=float)[dz_idx]
dz_lon = dz_table["lon"].to_numpy(dtype=float)[dz_idx]
pn_lat = pn_table["lat"].to_numpy(dtype=float)[pn_idx]
pn_lon = pn_table["lon"].to_numpy(dtype=float)[pn_idx]

dz_deg = dz_table["external_degree"].fillna(0).to_numpy(dtype=float)[dz_idx]
pn_deg = pn_table["external_degree"].fillna(0).to_numpy(dtype=float)[pn_idx]

# Vectorized attribute scores.
distance_km = haversine_numpy(dz_lat, dz_lon, pn_lat, pn_lon)
station_scores = batch_similarity(dz_station, pn_station)
corpus_scores = batch_similarity(dz_corpus, pn_corpus)
name_scores = np.maximum(station_scores, corpus_scores)
voltage_scores = np.where(np.isnan(dz_kv) | np.isnan(pn_kv), 0.50, (np.abs(dz_kv - pn_kv) <= SAME_KV_TOLERANCE).astype(np.float32))
geo_scores = geo_score_array(distance_km)
degree_scores = 1.0 - np.abs(dz_deg - pn_deg) / np.maximum(np.maximum(dz_deg, pn_deg), 1.0)

# Vectorized topology-profile scores.
neighbor_keys = profile_keys(dz_table["neighbor_kv_profile"], pn_table["neighbor_kv_profile"])
branch_keys = profile_keys(dz_table["branch_type_profile"], pn_table["branch_type_profile"])

dz_neighbor_mat = profile_matrix(dz_table, "neighbor_kv_profile", neighbor_keys)
pn_neighbor_mat = profile_matrix(pn_table, "neighbor_kv_profile", neighbor_keys)
dz_branch_mat = profile_matrix(dz_table, "branch_type_profile", branch_keys)
pn_branch_mat = profile_matrix(pn_table, "branch_type_profile", branch_keys)

neighbor_kv_scores = profile_similarity_indexed(dz_neighbor_mat, dz_idx, pn_neighbor_mat, pn_idx)
branch_type_scores = profile_similarity_indexed(dz_branch_mat, dz_idx, pn_branch_mat, pn_idx)

attribute_scores = (
    ATTRIBUTE_WEIGHTS["station_score"] * station_scores
    + ATTRIBUTE_WEIGHTS["name_score"] * name_scores
    + ATTRIBUTE_WEIGHTS["voltage_score"] * voltage_scores
    + ATTRIBUTE_WEIGHTS["geo_score"] * geo_scores
)

topology_score_pre = 0.34 * degree_scores + 0.33 * neighbor_kv_scores + 0.33 * branch_type_scores
preliminary_scores = FINAL_WEIGHTS["attribute_score"] * attribute_scores + FINAL_WEIGHTS["topology_score"] * topology_score_pre

group_candidate_scores = pd.DataFrame({
    "dayzer_group_id": group_candidates["dayzer_group_id"].values,
    "pano_group_id": group_candidates["pano_group_id"].values,
    "retrieval_score": group_candidates.get("retrieval_score", pd.Series([np.nan] * len(group_candidates))).values,
    "dz_station_norm": dz_station,
    "pn_station_norm": pn_station,
    "dz_kv": dz_kv,
    "pn_kv": pn_kv,
    "dz_member_count": dz_table["member_count"].to_numpy()[dz_idx],
    "pn_member_count": pn_table["member_count"].to_numpy()[pn_idx],
    "distance_km": distance_km,
    "station_score": station_scores,
    "name_score": name_scores,
    "voltage_score": voltage_scores,
    "geo_score": geo_scores,
    "degree_score": degree_scores,
    "neighbor_kv_score": neighbor_kv_scores,
    "branch_type_score": branch_type_scores,
    "matched_neighbor_score": 0.0,
    "attribute_score": attribute_scores,
    "topology_score_pre": topology_score_pre,
    "preliminary_score": preliminary_scores,
})

print("Scored group candidates:", group_candidate_scores.shape)
display(group_candidate_scores.sort_values("preliminary_score", ascending=False).head(10))


print(f"Step 9 elapsed time: {time.perf_counter() - t0_step9:.1f} seconds")


Scored group candidates: (1280213, 21)


,dayzer_group_id,pano_group_id,retrieval_score,dz_station_norm,pn_station_norm,dz_kv,pn_kv,dz_member_count,pn_member_count,distance_km,...,name_score,voltage_score,geo_score,degree_score,neighbor_kv_score,branch_type_score,matched_neighbor_score,attribute_score,topology_score_pre,preliminary_score
501756,DZG_LACY_CRK_345KV,PNG_LACY_CRK_345KV,1.0,LACY_CRK,LACY_CRK,345.0,345.0,1,1,1.054009,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
1029988,DZG_VICTPRT2_138KV,PNG_VICTPRT2_138KV,1.0,VICTPRT2,VICTPRT2,138.0,138.0,1,1,0.226702,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
1029653,DZG_VICTPORT_138KV,PNG_VICTPORT_138KV,1.0,VICTPORT,VICTPORT,138.0,138.0,1,1,0.027471,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
166262,DZG_CHAMON_138KV,PNG_CHAMON_138KV,1.0,CHAMON,CHAMON,138.0,138.0,1,1,0.629948,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
4906,DZG_AEEC_345KV,PNG_AEEC_345KV,1.0,AEEC,AEEC,345.0,345.0,1,1,1.093224,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
1088197,DZG_YNG_WND_345KV,PNG_YNG_WND_345KV,1.0,YNG_WND,YNG_WND,345.0,345.0,1,1,0.025088,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
76122,DZG_BENTSEN_138KV,PNG_BENTSEN_138KV,1.0,BENTSEN,BENTSEN,138.0,138.0,1,1,0.442723,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
1004406,DZG_TRUSGILL_345KV,PNG_TRUSGILL_345KV,1.0,TRUSGILL,TRUSGILL,345.0,345.0,1,1,0.470310,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
49024,DZG_AZURE_345KV,PNG_AZURE_345KV,1.0,AZURE,AZURE,345.0,345.0,1,1,0.066373,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
716970,DZG_PANDA_T2_345KV,PNG_PANDA_T2_345KV,1.0,PANDA_T2,PANDA_T2,345.0,345.0,1,1,0.128415,...,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0


Step 9 elapsed time: 46.6 seconds


## Step 10 — First-pass seeds and chunked matched-neighbor topology validation

This cell creates high-confidence seed group matches, then uses them to validate neighboring groups.

Logic:

1. First select very strong matches using attribute + topology evidence.
2. For each candidate $(d, p)$, check whether the already matched neighbors of Dayzer group $d$ land near Panorama group $p$.
3. If many seeded Dayzer neighbors map into Panorama neighbors of $p$, the candidate receives a higher topology score.

Matched-neighbor score:

$$
S_{mn}(d,p)=

rac{|\{m(n): n \in N(d), n 	ext{ has seed match}\} \cap N(p)|}
{|\{m(n): n \in N(d), n 	ext{ has seed match}\}|}
$$

Why this step can be slow:

- The broad-candidate version may evaluate over one million candidate pairs.
- This runtime setting splits the candidate pairs into chunks and computes the set intersections using `joblib` when available.


In [11]:
t0_step10 = time.perf_counter()

# First-pass best candidate per Dayzer group using preliminary score.
pre_sorted = group_candidate_scores.sort_values(
    ["dayzer_group_id", "preliminary_score"],
    ascending=[True, False]
).copy()

pre_sorted["pre_rank"] = pre_sorted.groupby("dayzer_group_id").cumcount() + 1
pre_best = pre_sorted[pre_sorted["pre_rank"] == 1].copy()
pre_second = pre_sorted[pre_sorted["pre_rank"] == 2][["dayzer_group_id", "preliminary_score"]].rename(
    columns={"preliminary_score": "second_preliminary_score"}
)
pre_best = pre_best.merge(pre_second, on="dayzer_group_id", how="left")
pre_best["pre_margin"] = pre_best["preliminary_score"] - pre_best["second_preliminary_score"].fillna(0)

# Conservative seed criteria.
seed_df = pre_best[
    (pre_best["preliminary_score"] >= 0.86)
    & (pre_best["pre_margin"] >= 0.08)
    & (pre_best["voltage_score"] >= 0.99)
    & ((pre_best["station_score"] >= 0.80) | (pre_best["geo_score"] >= 0.90))
].copy()

seed_map = dict(zip(seed_df["dayzer_group_id"], seed_df["pano_group_id"]))

print("High-confidence group seeds:", len(seed_map))
display(seed_df.sort_values("preliminary_score", ascending=False).head(10)[
    ["dayzer_group_id", "pano_group_id", "preliminary_score", "pre_margin", "station_score", "geo_score"]
])


# Precompute adjacency sets for group graphs.
dz_group_neighbors = {n: set(G_dz_group.neighbors(n)) for n in G_dz_group.nodes}
pn_group_neighbors = {n: set(G_pn_group.neighbors(n)) for n in G_pn_group.nodes}

# Faster matched-neighbor consistency:
# For each Dayzer group, precompute which of its neighbors already have seed Panorama mappings.
# Then each candidate only needs one small set intersection.
dz_mapped_neighbor_sets = {}
dz_mapped_neighbor_denoms = {}
for dz_gid, dz_nbrs in dz_group_neighbors.items():
    mapped = {seed_map[nbr] for nbr in dz_nbrs if nbr in seed_map}
    dz_mapped_neighbor_sets[dz_gid] = mapped
    dz_mapped_neighbor_denoms[dz_gid] = len(mapped)


def matched_neighbor_consistency_fast(dayzer_gid, pano_gid):
    """Measure whether seeded Dayzer neighbors map to neighbors of the Panorama candidate."""
    denom = dz_mapped_neighbor_denoms.get(dayzer_gid, 0)
    if denom == 0 or pd.isna(pano_gid):
        return 0.0
    mapped_neighbors = dz_mapped_neighbor_sets.get(dayzer_gid, set())
    pano_neighbors = pn_group_neighbors.get(pano_gid, set())
    if not mapped_neighbors or not pano_neighbors:
        return 0.0
    return len(mapped_neighbors & pano_neighbors) / denom


# Compute matched-neighbor score.
# This is Python set logic, so it is chunked and processed in chunks with joblib when available.
dz_candidate_gids = group_candidate_scores["dayzer_group_id"].to_numpy(dtype=object)
pn_candidate_gids = group_candidate_scores["pano_group_id"].to_numpy(dtype=object)


def matched_neighbor_score_slice(start, end):
    return np.array([
        matched_neighbor_consistency_fast(dz_candidate_gids[i], pn_candidate_gids[i])
        for i in range(start, end)
    ], dtype=np.float32)

ranges = [(i, min(i + MATCHED_NEIGHBOR_CHUNK_SIZE, len(group_candidate_scores)))
          for i in range(0, len(group_candidate_scores), MATCHED_NEIGHBOR_CHUNK_SIZE)]

if USE_PARALLEL_MATCHED_NEIGHBOR and len(ranges) > 1:
    print(f"Computing matched-neighbor scores in chunks: {len(ranges)} chunks, n_jobs={N_JOBS}")
    chunks = Parallel(n_jobs=N_JOBS, backend=PARALLEL_BACKEND, batch_size=1)(
        delayed(matched_neighbor_score_slice)(start, end)
        for start, end in ranges
    )
    group_candidate_scores["matched_neighbor_score"] = np.concatenate(chunks)
else:
    print("Computing matched-neighbor scores serially")
    group_candidate_scores["matched_neighbor_score"] = np.concatenate([
        matched_neighbor_score_slice(start, end) for start, end in ranges
    ])

group_candidate_scores["topology_score"] = (
    TOPOLOGY_WEIGHTS["degree_score"] * group_candidate_scores["degree_score"]
    + TOPOLOGY_WEIGHTS["neighbor_kv_score"] * group_candidate_scores["neighbor_kv_score"]
    + TOPOLOGY_WEIGHTS["branch_type_score"] * group_candidate_scores["branch_type_score"]
    + TOPOLOGY_WEIGHTS["matched_neighbor_score"] * group_candidate_scores["matched_neighbor_score"]
)

group_candidate_scores["final_group_score"] = (
    FINAL_WEIGHTS["attribute_score"] * group_candidate_scores["attribute_score"]
    + FINAL_WEIGHTS["topology_score"] * group_candidate_scores["topology_score"]
)

display(group_candidate_scores.sort_values("final_group_score", ascending=False).head(10))


print(f"Step 10 elapsed time: {time.perf_counter() - t0_step10:.1f} seconds")


High-confidence group seeds: 3214


,dayzer_group_id,pano_group_id,preliminary_score,pre_margin,station_score,geo_score
1461,DZG_CBECII_345KV,PNG_CBECII_345KV,1.0,0.362500,1.0,1.0
1569,DZG_CHALUPA_138KV,PNG_CHALUPA_138KV,1.0,0.229615,1.0,1.0
9567,DZG_TRUSGILL_345KV,PNG_TRUSGILL_345KV,1.0,0.250096,1.0,1.0
1314,DZG_CALHOUN_138KV,PNG_CALHOUN_138KV,1.0,0.369423,1.0,1.0
9381,DZG_TNTEPPCO_69KV,PNG_TNTEPPCO_69KV,1.0,0.165000,1.0,1.0
1583,DZG_CHAMON_138KV,PNG_CHAMON_138KV,1.0,0.281133,1.0,1.0
1668,DZG_CHISMGRD_138KV,PNG_CHISMGRD_138KV,1.0,0.331000,1.0,1.0
6134,DZG_MISAE_345KV,PNG_MISAE_345KV,1.0,0.269423,1.0,1.0
10338,DZG_WSHACK_345KV,PNG_WSHACK_345KV,1.0,0.352500,1.0,1.0
6099,DZG_MILHWY_138KV,PNG_MILHWY_138KV,1.0,0.255714,1.0,1.0


Computing matched-neighbor scores in chunks: 9 chunks, n_jobs=16


,dayzer_group_id,pano_group_id,retrieval_score,dz_station_norm,pn_station_norm,dz_kv,pn_kv,dz_member_count,pn_member_count,distance_km,...,geo_score,degree_score,neighbor_kv_score,branch_type_score,matched_neighbor_score,attribute_score,topology_score_pre,preliminary_score,topology_score,final_group_score
316182,DZG_E_BLACK_138KV,PNG_E_BLACK_138KV,1.0,E_BLACK,E_BLACK,138.0,138.0,1,1,0.905971,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1066022,DZG_WINDMILL_345KV,PNG_WINDMILL_345KV,1.0,WINDMILL,WINDMILL,345.0,345.0,1,1,0.117796,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
384238,DZG_GRBY8_138KV,PNG_GRBY8_138KV,1.0,GRBY8,GRBY8,138.0,138.0,1,1,0.451426,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
325443,DZG_FERMI_138KV,PNG_FERMI_138KV,1.0,FERMI,FERMI,138.0,138.0,1,1,0.906428,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
709258,DZG_OXY_CC_138KV,PNG_OXY_CC_138KV,1.0,OXY_CC,OXY_CC,138.0,138.0,1,1,0.927216,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
289875,DZG_ELLISSLR_138KV,PNG_ELLISSLR_138KV,1.0,ELLISSLR,ELLISSLR,138.0,138.0,1,1,0.749666,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1013433,DZG_TYLRWIND_138KV,PNG_TYLRWIND_138KV,1.0,TYLRWIND,TYLRWIND,138.0,138.0,1,1,1.777641,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
286605,DZG_ELARA_SL_138KV,PNG_ELARA_SL_138KV,1.0,ELARA_SL,ELARA_SL,138.0,138.0,1,1,0.021470,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
988257,DZG_TNWILLOW_345KV,PNG_TNWILLOW_345KV,1.0,TNWILLOW,TNWILLOW,345.0,345.0,1,1,1.398892,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
756009,DZG_PNPI_138KV,PNG_PNPI_138KV,1.0,PNPI,PNPI,138.0,138.0,1,1,0.767599,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


Step 10 elapsed time: 2.0 seconds


## Step 11 — Select group-level matches and assign confidence

This cell selects the best Panorama group for each Dayzer group and assigns status labels.

Why this step matters:

- This method separates **match confidence** from **review flags**.
- We do not mark non-winners as `conflict_lost`.
- We preserve possible many-to-one and one-to-many situations for review.

In [12]:
group_ranked = group_candidate_scores.sort_values(
    ["dayzer_group_id", "final_group_score"],
    ascending=[True, False]
).copy()

group_ranked["candidate_rank"] = group_ranked.groupby("dayzer_group_id").cumcount() + 1

group_best = group_ranked[group_ranked["candidate_rank"] == 1].copy()
group_second = (
    group_ranked[group_ranked["candidate_rank"] == 2][["dayzer_group_id", "final_group_score"]]
    .rename(columns={"final_group_score": "second_group_score"})
)
group_best = group_best.merge(group_second, on="dayzer_group_id", how="left")
group_best["group_score_margin"] = group_best["final_group_score"] - group_best["second_group_score"].fillna(0)

group_best["group_match_status"] = [
    score_status(s, m) for s, m in zip(group_best["final_group_score"], group_best["group_score_margin"])
]

# Add unmatched Dayzer groups that had no candidate.
all_dz_groups = set(dayzer_group["group_id"])
matched_dz_groups = set(group_best["dayzer_group_id"])
missing_groups = sorted(all_dz_groups - matched_dz_groups)
if missing_groups:
    missing_rows = []
    for gid in missing_groups:
        drow = dayzer_group[dayzer_group["group_id"] == gid].iloc[0]
        missing_rows.append({
            "dayzer_group_id": gid,
            "pano_group_id": np.nan,
            "dz_station_norm": drow["station_norm"],
            "pn_station_norm": "",
            "dz_kv": drow["kv"],
            "pn_kv": np.nan,
            "dz_member_count": drow["member_count"],
            "pn_member_count": np.nan,
            "dz_member_buses": drow["member_buses"],
            "pn_member_buses": [],
            "distance_km": np.nan,
            "station_score": 0.0,
            "name_score": 0.0,
            "voltage_score": 0.0,
            "geo_score": 0.0,
            "degree_score": 0.0,
            "neighbor_kv_score": 0.0,
            "branch_type_score": 0.0,
            "matched_neighbor_score": 0.0,
            "attribute_score": 0.0,
            "topology_score": 0.0,
            "final_group_score": 0.0,
            "second_group_score": np.nan,
            "group_score_margin": 0.0,
            "group_match_status": "unmatched",
        })
    group_best = pd.concat([group_best, pd.DataFrame(missing_rows)], ignore_index=True)

# Review flags
pano_claim_counts = group_best["pano_group_id"].value_counts(dropna=True).to_dict()
group_best["pano_group_claim_count"] = group_best["pano_group_id"].map(pano_claim_counts).fillna(0).astype(int)
group_best["many_dayzer_groups_to_one_pano_group"] = group_best["pano_group_claim_count"] > 1
group_best["multi_member_group_flag"] = (
    (group_best["dz_member_count"].fillna(0) > 1)
    | (group_best["pn_member_count"].fillna(0) > 1)
)
group_best["needs_review"] = (
    group_best["group_match_status"].isin(["ambiguous", "matched_low", "unmatched"])
    | group_best["many_dayzer_groups_to_one_pano_group"]
    | group_best["multi_member_group_flag"]
)

print("Group-level match status counts:")
display(group_best["group_match_status"].value_counts(dropna=False).to_frame("count"))

print("\nGroup-level review flags:")
display(group_best[["many_dayzer_groups_to_one_pano_group", "multi_member_group_flag", "needs_review"]].sum().to_frame("count"))

display(group_best.sort_values("final_group_score", ascending=False).head(10)[
    ["dayzer_group_id", "pano_group_id", "final_group_score", "group_score_margin", "group_match_status", "needs_review"]
])

Group-level match status counts:


,count
group_match_status,
matched_high,3174
ambiguous,2962
matched_medium,2755
matched_low,1325
unmatched,225



Group-level review flags:


,count
many_dayzer_groups_to_one_pano_group,5848
multi_member_group_flag,3160
needs_review,6985


,dayzer_group_id,pano_group_id,final_group_score,group_score_margin,group_match_status,needs_review
10338,DZG_WSHACK_345KV,PNG_WSHACK_345KV,1.0,0.425000,matched_high,False
956,DZG_BORDAS_345KV,PNG_BORDAS_345KV,1.0,0.270385,matched_high,False
7181,DZG_PNPI_138KV,PNG_PNPI_138KV,1.0,0.353095,matched_high,True
7101,DZG_PISGAH_345KV,PNG_PISGAH_345KV,1.0,0.352500,matched_high,False
6310,DZG_MWEC_138KV,PNG_MWEC_138KV,1.0,0.325000,matched_high,False
1461,DZG_CBECII_345KV,PNG_CBECII_345KV,1.0,0.462500,matched_high,False
10135,DZG_WHIT_RVR_345KV,PNG_WHIT_RVR_345KV,1.0,0.450192,matched_high,True
10074,DZG_WHCCS2_345KV,PNG_WHCCS2_345KV,1.0,0.241429,matched_high,True
1353,DZG_CAMWIND_138KV,PNG_CAMWIND_138KV,1.0,0.211000,matched_high,False
1314,DZG_CALHOUN_138KV,PNG_CALHOUN_138KV,1.0,0.393333,matched_high,False


## Step 12 — Refine from group-level matches to raw bus-level output

This cell produces one output row per Dayzer raw bus.

Refinement logic:

- If a matched Dayzer group and Panorama group both contain one raw bus, the result is an `exact_bus` match.
- If a group contains multiple raw buses, the notebook still proposes a best Panorama raw bus, but marks the mapping as `ambiguous_within_group` or `group_level`.
- This avoids pretending that group-level evidence uniquely identifies every raw bus.

Why this step matters:

- The required CSV asks for `dayzer_bus` and `pano_bus`.
- provides those columns while also telling the reviewer whether the result is exact or group-level.

In [13]:
#  bus-level output is derived from the group-level match.
# For multi-member Panorama groups, we provide a representative Panorama bus but mark
# the mapping as ambiguous/group-level rather than claiming exact raw-bus certainty.

# Representative Panorama bus per group.
# The representative is used only to populate the required `pano_bus` column.
# It is not treated as authoritative when the group contains multiple Panorama buses.
pano_group_rep = (
    pano_bus.sort_values(["group_id", "bus_name"])
    .groupby("group_id", as_index=False)
    .agg(
        pano_group_id=("group_id", "first"),
        pano_bus=("bus_name", "first"),
        pano_kv=("kv", "first"),
        pano_lat=("lat", "first"),
        pano_lon=("lon", "first"),
        pano_group_member_count=("bus_name", "size"),
        pano_group_members=("bus_name", lambda x: sorted(x.astype(str).tolist())),
    )
)

# Prepare group-level match table for merging onto each Dayzer raw bus.
group_cols_for_bus = [
    "dayzer_group_id", "pano_group_id",
    "dz_member_count", "pn_member_count",
    "distance_km",
    "station_score", "name_score", "voltage_score", "geo_score",
    "degree_score", "neighbor_kv_score", "branch_type_score", "matched_neighbor_score",
    "topology_score", "final_group_score", "group_score_margin",
    "group_match_status",
    "many_dayzer_groups_to_one_pano_group", "multi_member_group_flag", "needs_review",
]

group_for_bus = group_best[group_cols_for_bus].copy()

bus_mapping_results = (
    dayzer_bus.rename(columns={
        "bus_name": "dayzer_bus",
        "kv": "dayzer_kv",
        "lat": "dayzer_lat",
        "lon": "dayzer_lon",
    })
    .merge(group_for_bus, left_on="group_id", right_on="dayzer_group_id", how="left")
    .merge(pano_group_rep, on="pano_group_id", how="left")
)

# Remove large list-valued helper columns before row-wise interpretation.
if "pano_group_members" in bus_mapping_results.columns:
    bus_mapping_results = bus_mapping_results.drop(columns=["pano_group_members"])
if "group_id" in bus_mapping_results.columns:
    bus_mapping_results = bus_mapping_results.drop(columns=["group_id"])

# Bus-level representative score:
# For exact one-to-one groups, this is a meaningful raw-bus score.
# For multi-member groups, it is only a representative score and should not be over-interpreted.
bus_mapping_results["bus_name_score"] = [
    fast_similarity(dz, pn) if pd.notna(pn) else 0.0
    for dz, pn in zip(bus_mapping_results["name_norm"], bus_mapping_results["pano_bus"])
]
bus_mapping_results["bus_station_score"] = [
    fast_similarity(dz, pn) if pd.notna(pn) else 0.0
    for dz, pn in zip(bus_mapping_results["station_norm"], bus_mapping_results["pano_bus"])
]
bus_mapping_results["bus_distance_km"] = [
    haversine_km(dla, dlo, pla, plo)
    for dla, dlo, pla, plo in zip(
        bus_mapping_results["dayzer_lat"],
        bus_mapping_results["dayzer_lon"],
        bus_mapping_results["pano_lat"],
        bus_mapping_results["pano_lon"],
    )
]
bus_mapping_results["bus_geo_score"] = bus_mapping_results["bus_distance_km"].map(geo_score_from_distance)

bus_mapping_results["bus_level_score"] = (
    0.40 * bus_mapping_results["bus_name_score"].fillna(0)
    + 0.25 * bus_mapping_results["bus_station_score"].fillna(0)
    + 0.35 * bus_mapping_results["bus_geo_score"].fillna(0)
)
bus_mapping_results["bus_score_margin"] = np.nan

def assign_mapping_level(row):
    """Assign raw-bus vs group-level mapping interpretation."""
    if row.get("group_match_status") == "unmatched" or pd.isna(row.get("pano_group_id")):
        return "unmatched"
    if row.get("dz_member_count", 0) == 1 and row.get("pn_member_count", 0) == 1:
        return "exact_bus"
    if row.get("pn_member_count", 0) > 1 or row.get("dz_member_count", 0) > 1:
        return "ambiguous_within_group"
    return "group_level"

bus_mapping_results["mapping_level"] = bus_mapping_results.apply(assign_mapping_level, axis=1)
bus_mapping_results["confidence"] = bus_mapping_results["group_match_status"].fillna("unmatched")

# Any non-exact mapping should be reviewed.
bus_mapping_results["needs_review"] = (
    bus_mapping_results["needs_review"].fillna(True)
    | (bus_mapping_results["mapping_level"] != "exact_bus")
)

# Rename group distance to keep final schema consistent.
bus_mapping_results["distance_km"] = bus_mapping_results["bus_distance_km"].combine_first(bus_mapping_results["distance_km"])

# No expensive raw bus candidate detail is generated in  by default.
# The group-level candidate file is the main diagnostic artifact.
bus_candidate_scores = pd.DataFrame()

print("Bus-level output rows:", bus_mapping_results.shape)
print("\\nMapping level counts:")
display(bus_mapping_results["mapping_level"].value_counts(dropna=False).to_frame("count"))

print("\\nConfidence counts:")
display(bus_mapping_results["confidence"].value_counts(dropna=False).to_frame("count"))

display(bus_mapping_results[[
    "dayzer_bus", "pano_bus", "dayzer_group_id", "pano_group_id",
    "mapping_level", "confidence", "final_group_score", "needs_review"
]].head(10))

Bus-level output rows: (10558, 52)
\nMapping level counts:


,count
mapping_level,
exact_bus,7181
ambiguous_within_group,3149
unmatched,228


\nConfidence counts:


,count
confidence,
matched_high,3213
ambiguous,2975
matched_medium,2807
matched_low,1335
unmatched,228


,dayzer_bus,pano_bus,dayzer_group_id,pano_group_id,mapping_level,confidence,final_group_score,needs_review
0,1,FOWLRTON_1KV_1,DZG_1_1KV,PNG_FOWLRTON_1KV,exact_bus,matched_low,0.641000,True
1,1000,FAYETT_1KV_1,DZG_1000_1KV,PNG_FAYETT_1KV,exact_bus,matched_low,0.670000,True
2,111138,BAC_1KV_1,DZG_111138_1KV,PNG_BAC_1KV,unmatched,unmatched,0.489286,True
3,111139_STAR,BAC_1KV_1,DZG_111139_STAR_1KV,PNG_BAC_1KV,ambiguous_within_group,ambiguous,0.515000,True
4,111140_STAR,BAC_1KV_1,DZG_111140_STAR_1KV,PNG_BAC_1KV,ambiguous_within_group,ambiguous,0.515000,True
5,112420,RAB_1KV_1,DZG_112420_1KV,PNG_RAB_1KV,unmatched,unmatched,0.466333,True
6,112421,RAB_1KV_1,DZG_112421_1KV,PNG_RAB_1KV,unmatched,unmatched,0.466333,True
7,112422,RAB_1KV_1,DZG_112422_1KV,PNG_RAB_1KV,unmatched,unmatched,0.466333,True
8,112423,RAB_1KV_1,DZG_112423_1KV,PNG_RAB_1KV,unmatched,unmatched,0.466333,True
9,112880,AZURE_1KV_1,DZG_112880_1KV,PNG_AZURE_1KV,unmatched,unmatched,0.455735,True


## Step 13 — Add human-readable reason codes

This cell adds a `reason` column explaining why a match was labeled the way it was.

Why this step matters:

- In a real review workflow, analysts need to understand why a mapping is high-confidence, ambiguous, or group-level.
- A good mapping table should be interpretable, not just numerical.

In [14]:
# Add human-readable reason codes.
# This cell explains why a row is considered exact, group-level, ambiguous, or review-needed.
# The reason column is useful for analysts because it makes the matching decision auditable.

reasons = []
for r in bus_mapping_results.itertuples(index=False):
    parts = []

    mapping_level = getattr(r, "mapping_level", "")
    confidence = getattr(r, "confidence", "")
    station_score = getattr(r, "station_score", np.nan)
    geo_score = getattr(r, "geo_score", np.nan)
    topology_score = getattr(r, "topology_score", np.nan)
    matched_neighbor_score = getattr(r, "matched_neighbor_score", 0.0)
    many_to_one = bool(getattr(r, "many_dayzer_groups_to_one_pano_group", False))
    multi_member = bool(getattr(r, "multi_member_group_flag", False))

    if mapping_level == "exact_bus":
        parts.append("one-to-one station-kV group; representative raw bus mapping")
    elif mapping_level == "ambiguous_within_group":
        parts.append("group-level match is plausible but exact raw bus is ambiguous")
    elif mapping_level == "unmatched":
        parts.append("no reliable group-level candidate")
    else:
        parts.append(str(mapping_level))

    if pd.notna(station_score) and station_score >= 0.85:
        parts.append("strong station/name similarity")
    elif pd.notna(station_score) and station_score < 0.50:
        parts.append("weak station/name similarity")

    if pd.notna(geo_score) and geo_score >= 0.90:
        parts.append("close geography")
    elif pd.notna(geo_score) and geo_score <= 0.35:
        parts.append("weak or missing geography")

    if pd.notna(topology_score) and topology_score >= 0.75:
        parts.append("topology consistent")
    elif pd.notna(matched_neighbor_score) and matched_neighbor_score > 0:
        parts.append("some matched-neighbor support")

    if many_to_one:
        parts.append("many Dayzer groups claim same Panorama group; review needed")

    if multi_member:
        parts.append("multi-member group indicates possible split/merge granularity difference")

    if confidence == "ambiguous":
        parts.append("top candidate margin is small")

    reasons.append("; ".join(parts))

bus_mapping_results["reason"] = reasons

display(bus_mapping_results[[
    "dayzer_bus", "pano_bus", "mapping_level", "confidence", "final_group_score", "bus_level_score", "needs_review", "reason"
]].head(15))


,dayzer_bus,pano_bus,mapping_level,confidence,final_group_score,bus_level_score,needs_review,reason
0,1,FOWLRTON_1KV_1,exact_bus,matched_low,0.641000,0.331667,True,one-to-one station-kV group; representative ra...
1,1000,FAYETT_1KV_1,exact_bus,matched_low,0.670000,0.396250,True,one-to-one station-kV group; representative ra...
2,111138,BAC_1KV_1,unmatched,unmatched,0.489286,0.488333,True,no reliable group-level candidate; weak statio...
3,111139_STAR,BAC_1KV_1,ambiguous_within_group,ambiguous,0.515000,0.445000,True,group-level match is plausible but exact raw b...
4,111140_STAR,BAC_1KV_1,ambiguous_within_group,ambiguous,0.515000,0.445000,True,group-level match is plausible but exact raw b...
5,112420,RAB_1KV_1,unmatched,unmatched,0.466333,0.523333,True,no reliable group-level candidate; weak statio...
6,112421,RAB_1KV_1,unmatched,unmatched,0.466333,0.523333,True,no reliable group-level candidate; weak statio...
7,112422,RAB_1KV_1,unmatched,unmatched,0.466333,0.523333,True,no reliable group-level candidate; weak statio...
8,112423,RAB_1KV_1,unmatched,unmatched,0.466333,0.523333,True,no reliable group-level candidate; weak statio...
9,112880,AZURE_1KV_1,unmatched,unmatched,0.455735,0.170441,True,no reliable group-level candidate; weak statio...


## Step 14 — Conflict and granularity diagnostics

This cell creates diagnostic tables for many-to-one and multi-member group situations.

Why this step matters:

- Here, apparent conflicts are not automatically treated as failures.
- They may reveal true model-granularity differences, such as one dataset splitting a station-kV topology bus more finely than the other.

In [15]:
# Panorama group claims by multiple Dayzer groups
group_conflicts = (
    group_best.dropna(subset=["pano_group_id"])
    .groupby("pano_group_id")
    .agg(
        dayzer_group_count=("dayzer_group_id", "nunique"),
        dayzer_groups=("dayzer_group_id", lambda x: sorted(set(x))),
        best_score=("final_group_score", "max"),
        avg_score=("final_group_score", "mean"),
    )
    .reset_index()
    .query("dayzer_group_count > 1")
    .sort_values(["dayzer_group_count", "best_score"], ascending=[False, False])
)

# Multi-member group matches
multi_member_matches = group_best[
    (group_best["dz_member_count"].fillna(0) > 1)
    | (group_best["pn_member_count"].fillna(0) > 1)
].copy()

print("Group conflicts:", group_conflicts.shape)
display(group_conflicts.head(20))

print("\nMulti-member group matches:", multi_member_matches.shape)
display(multi_member_matches[[
    "dayzer_group_id", "pano_group_id", "dz_member_count", "pn_member_count",
    "final_group_score", "group_match_status", "needs_review"
]].sort_values(["dz_member_count", "pn_member_count"], ascending=False).head(20))

Group conflicts: (1944, 5)


,pano_group_id,dayzer_group_count,dayzer_groups,best_score,avg_score
1653,PNG_DPW_1KV,40,"[DZG_49114_1KV, DZG_ADICKS_A1STR_1KV, DZG_ADIC...",0.711143,0.557315
1654,PNG_DPW_23KV,39,"[DZG_49079_23KV, DZG_ADICKS_A1TER_23KV, DZG_AD...",0.708545,0.548748
3773,PNG_MIRAGE_14KV,25,"[DZG_AZ_AZ_G1_14KV, DZG_AZ_AZ_G2_14KV, DZG_AZ_...",0.934000,0.620122
4399,PNG_PES1_138KV,20,"[DZG_ALEXIS84T1_138KV, DZG_BELAIR_SVC_138KV, D...",0.900000,0.645140
2646,PNG_HI_LONEB_34KV,19,"[DZG_HILO_4_5_C2_34KV, DZG_HILO_4_5_C3_34KV, D...",0.947895,0.554934
974,PNG_CDHSW_14KV,19,"[DZG_CHILSW1_14KV, DZG_DOW_A2_14KV, DZG_DOW_A3...",0.890000,0.559228
1340,PNG_CRLNW_14KV,18,"[DZG_ALLENSW1_14KV, DZG_ALLENSW2_14KV, DZG_CNV...",0.890714,0.637319
6015,PNG_UCC_COGN_14KV,18,"[DZG_AMO_AMOCO_S1_14KV, DZG_BAC_CTG1_14KV, DZG...",0.719118,0.627507
5253,PNG_SHAMROCK_34KV,18,"[DZG_BEARKAT_C1_34KV, DZG_BEARKAT_C2_34KV, DZG...",0.702895,0.538565
4613,PNG_PTCOMFOR_69KV,16,"[DZG_FORMOSA_15_69KV, DZG_FORMOSA_17_69KV, DZG...",0.811667,0.610420



Multi-member group matches: (3160, 31)


,dayzer_group_id,pano_group_id,dz_member_count,pn_member_count,final_group_score,group_match_status,needs_review
8437,DZG_SL_12KV,PNG_SL_12KV,7,3,0.763636,matched_medium,True
2432,DZG_DOWGEN_14KV,PNG_DOWGEN_14KV,6,19,0.744456,matched_medium,True
8438,DZG_SL_34KV,PNG_SL_34KV,5,2,0.766667,matched_medium,True
4166,DZG_HUD_8_138KV,PNG_HUD_138KV,4,5,0.802500,matched_medium,True
6373,DZG_NA_12KV,PNG_NA_12KV,4,5,0.805000,matched_medium,True
964,DZG_BOSQ_BSQSU_18KV,PNG_BOSQUESW_18KV,4,4,0.853333,matched_high,True
6704,DZG_OLIN_OLING_14KV,PNG_OLINGR_14KV,4,4,0.876250,matched_high,True
7426,DZG_QALSW_1KV,PNG_QALSW_1KV,4,4,1.000000,matched_high,True
8192,DZG_SCLP_SCLPC_14KV,PNG_SCLPCOGN_14KV,4,4,0.853333,matched_high,True
2223,DZG_DECKE_DPGT_14KV,PNG_DECKER_14KV,4,2,0.776250,matched_medium,True


## Step 15 — Local graph visualization for case studies

This cell defines a function to plot local raw-bus neighborhoods.

Why this step matters:

- Full network graphs with 10,000+ nodes are not visually useful.
- Local 1-hop or 2-hop neighborhoods help validate whether two matched buses occupy similar electrical positions.
- This is especially useful for ambiguous or topology-supported matches.

Use examples:

```python
plot_local_match("KEITHSW", "KEITHSW_138KV_1", radius=1)
```

In [16]:
def ego_nodes(G, center, radius=1):
    """Return nodes in an ego neighborhood around center."""
    if center not in G:
        return set()
    lengths = nx.single_source_shortest_path_length(G, center, cutoff=radius)
    return set(lengths.keys())


def plot_local_match(dayzer_bus_name, pano_bus_name, radius=1):
    """Plot Dayzer and Panorama local neighborhoods side by side."""
    dz_nodes = ego_nodes(G_dz_raw, dayzer_bus_name, radius=radius)
    pn_nodes = ego_nodes(G_pn_raw, pano_bus_name, radius=radius)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    if dz_nodes:
        H_dz = G_dz_raw.subgraph(dz_nodes)
        pos = nx.spring_layout(H_dz, seed=7)
        nx.draw_networkx(H_dz, pos=pos, with_labels=True, font_size=8, ax=axes[0], node_size=500)
        axes[0].set_title(f"Dayzer local graph: {dayzer_bus_name}")
    else:
        axes[0].text(0.5, 0.5, "Dayzer bus not found", ha="center")
        axes[0].set_title("Dayzer local graph")

    if pn_nodes:
        H_pn = G_pn_raw.subgraph(pn_nodes)
        pos = nx.spring_layout(H_pn, seed=7)
        nx.draw_networkx(H_pn, pos=pos, with_labels=True, font_size=8, ax=axes[1], node_size=500)
        axes[1].set_title(f"Panorama local graph: {pano_bus_name}")
    else:
        axes[1].text(0.5, 0.5, "Panorama bus not found", ha="center")
        axes[1].set_title("Panorama local graph")

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# Display a few high-confidence exact or near-exact examples if available.
example_rows = bus_mapping_results[
    (bus_mapping_results["confidence"].isin(["matched_high", "matched_medium"]))
    & (bus_mapping_results["pano_bus"].notna())
].head(3)

display(example_rows[["dayzer_bus", "pano_bus", "mapping_level", "confidence", "reason"]])

# Uncomment the following lines to plot examples interactively:
# for _, ex in example_rows.iterrows():
#     plot_local_match(ex["dayzer_bus"], ex["pano_bus"], radius=1)

,dayzer_bus,pano_bus,mapping_level,confidence,reason
51,7RNCHSLR_8,7RNCHSLR_138KV_1,exact_bus,matched_high,one-to-one station-kV group; representative ra...
52,7RNCHSLR_S1,7RNCHSLR_34KV_1,ambiguous_within_group,matched_medium,group-level match is plausible but exact raw b...
53,7RNCHSLR_S2,7RNCHSLR_34KV_1,ambiguous_within_group,matched_medium,group-level match is plausible but exact raw b...


## Step 16 — Summary diagnostics

This cell summarizes the output.

Why this step matters:

- The assignment values the approach and reasoning.
- These diagnostics show coverage, confidence, ambiguity, and review workload.

In [21]:
summary_tables = {}

summary_tables["mapping_level_counts"] = bus_mapping_results["mapping_level"].value_counts(dropna=False).to_frame("count")
summary_tables["confidence_counts"] = bus_mapping_results["confidence"].value_counts(dropna=False).to_frame("count")
summary_tables["needs_review_counts"] = bus_mapping_results["needs_review"].value_counts(dropna=False).to_frame("count")
summary_tables["multi_member_flag_counts"] = bus_mapping_results["multi_member_group_flag"].value_counts(dropna=False).to_frame("count")

print("Mapping level counts:")
display(summary_tables["mapping_level_counts"])

print("\nConfidence counts:")
display(summary_tables["confidence_counts"])

print("\nNeeds review counts:")
display(summary_tables["needs_review_counts"])

print("\nDistance summary for matched rows:")
display(bus_mapping_results.loc[bus_mapping_results["pano_bus"].notna(), "distance_km"].describe().to_frame("distance_km"))

print("\nHigh-confidence examples:")
display(bus_mapping_results[bus_mapping_results["confidence"] == "matched_high"].head(10)[
    ["dayzer_bus", "pano_bus", "mapping_level", "final_group_score", "bus_level_score", "reason"]
])

print("\nAmbiguous/group-level examples:")
display(bus_mapping_results[bus_mapping_results["needs_review"]].head(10)[
    ["dayzer_bus", "pano_bus", "mapping_level", "confidence", "final_group_score", "reason"]
])

Mapping level counts:


,count
mapping_level,
exact_bus,7181
ambiguous_within_group,3149
unmatched,228



Confidence counts:


,count
confidence,
matched_high,3213
ambiguous,2975
matched_medium,2807
matched_low,1335
unmatched,228



Needs review counts:


...

,distance_km
count,10112.000000
mean,33.263403
std,99.954673
min,0.003303
25%,2.503052
50%,8.653273
75%,18.542797
max,865.940928



High-confidence examples:


,dayzer_bus,pano_bus,mapping_level,final_group_score,bus_level_score,reason
51,7RNCHSLR_8,7RNCHSLR_138KV_1,exact_bus,0.855000,0.719359,one-to-one station-kV group; representative ra...
60,ABBYBEND,ABBYBEND_138KV_1,exact_bus,0.985000,0.748333,one-to-one station-kV group; representative ra...
61,ABDY1TAP2A,ABDY1TAP_69KV_1,exact_bus,0.918333,0.661000,one-to-one station-kV group; representative ra...
62,ABEAST2A,ABEAST_69KV_1,ambiguous_within_group,0.852857,0.721429,group-level match is plausible but exact raw b...
64,ABEAST4A,ABEAST_138KV_1,ambiguous_within_group,0.952857,0.704545,group-level match is plausible but exact raw b...
65,ABERNATH,ABERNATH_345KV_1,exact_bus,0.855000,0.678333,one-to-one station-kV group; representative ra...
70,ABLSPRRC,ABLSPRRC_138KV_1,exact_bus,0.855000,0.678333,one-to-one station-kV group; representative ra...
73,ABNTHWST2A,ABNTHWST_69KV_1,exact_bus,0.863333,0.766000,one-to-one station-kV group; representative ra...
74,ABNTHWST2B,ABNTHWST_69KV_1,exact_bus,0.863333,0.766000,one-to-one station-kV group; representative ra...
75,ABNTHWST4A,ABNTHWST_138KV_1,ambiguous_within_group,0.877619,0.750000,group-level match is plausible but exact raw b...



Ambiguous/group-level examples:


,dayzer_bus,pano_bus,mapping_level,confidence,final_group_score,reason
0,1,FOWLRTON_1KV_1,exact_bus,matched_low,0.641000,one-to-one station-kV group; representative ra...
1,1000,FAYETT_1KV_1,exact_bus,matched_low,0.670000,one-to-one station-kV group; representative ra...
2,111138,BAC_1KV_1,unmatched,unmatched,0.489286,no reliable group-level candidate; weak statio...
3,111139_STAR,BAC_1KV_1,ambiguous_within_group,ambiguous,0.515000,group-level match is plausible but exact raw b...
4,111140_STAR,BAC_1KV_1,ambiguous_within_group,ambiguous,0.515000,group-level match is plausible but exact raw b...
5,112420,RAB_1KV_1,unmatched,unmatched,0.466333,no reliable group-level candidate; weak statio...
6,112421,RAB_1KV_1,unmatched,unmatched,0.466333,no reliable group-level candidate; weak statio...
7,112422,RAB_1KV_1,unmatched,unmatched,0.466333,no reliable group-level candidate; weak statio...
8,112423,RAB_1KV_1,unmatched,unmatched,0.466333,no reliable group-level candidate; weak statio...
9,112880,AZURE_1KV_1,unmatched,unmatched,0.455735,no reliable group-level candidate; weak statio...


# Step 17 - Export the required matched-result CSV

This cell writes the final matched-result file required by the assignment. It uses `bus_mapping_results` from the pipeline and aliases it to `final_bus_results` for the final export.

Output file:

- `assignment3_matched_results.csv`



In [25]:
# ============================================================
# Step 17 - Export the required matched-result CSV
# ============================================================

from pathlib import Path
import pandas as pd


def _find_assignment3_dir() -> Path:
    candidates = [
        Path("."),
        Path(".."),
        Path("assignment3_bus_mapping"),
        Path("ecesis-assignment-submission/assignment3_bus_mapping"),
        Path("../ecesis-assignment-submission/assignment3_bus_mapping"),
        Path("../../ecesis-assignment-submission/assignment3_bus_mapping"),
        Path("Summer2026-main/ecesis-assignment-submission/assignment3_bus_mapping"),
    ]
    for candidate in candidates:
        if (candidate / "outputs").exists() and (candidate / "notebook").exists():
            return candidate
    return Path("..")


assignment3_dir = globals().get("ASSIGNMENT_DIR", _find_assignment3_dir())
OUTPUT_DIR = Path(globals().get("OUT_DIR", Path(assignment3_dir) / "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_CSV = OUTPUT_DIR / "assignment3_matched_results.csv"
legacy_result_csv = OUTPUT_DIR / "assignment3_bus_mapping_results.csv"

if "final_bus_results" in globals():
    final_bus_results = final_bus_results.copy()
elif "bus_mapping_results" in globals():
    final_bus_results = bus_mapping_results.copy()
elif RESULT_CSV.exists():
    final_bus_results = pd.read_csv(RESULT_CSV)
elif legacy_result_csv.exists():
    final_bus_results = pd.read_csv(legacy_result_csv)
else:
    raise NameError(
        "Neither final_bus_results nor bus_mapping_results is defined, and no existing result CSV was found. "
        "Run the matching pipeline cells before exporting."
    )

required_output_columns = ["dayzer_bus", "pano_bus"]
missing_columns = [col for col in required_output_columns if col not in final_bus_results.columns]
if missing_columns:
    raise ValueError(f"Final Assignment 3 output is missing required columns: {missing_columns}")

final_bus_results.to_csv(RESULT_CSV, index=False)

print(f"Saved final matched-result CSV to: {RESULT_CSV}")
print(f"Rows: {len(final_bus_results):,}")
print(f"Columns: {len(final_bus_results.columns):,}")

display(final_bus_results.head(20))





Saved final matched-result CSV to: G:\Users\wentao\Desktop\ECEsis\Summer2026-main\ecesis-assignment-submission\assignment3_bus_mapping\outputs\assignment3_matched_results.csv
Rows: 10,558
Columns: 30


,dayzer_bus,pano_bus,dayzer_group_id,pano_group_id,mapping_level,dayzer_kv,pano_kv,dayzer_lat,dayzer_lon,pano_lat,...,topology_score,final_group_score,group_score_margin,bus_level_score,bus_score_margin,confidence,many_dayzer_groups_to_one_pano_group,multi_member_group_flag,needs_review,reason
0,1,FOWLRTON_1KV_1,DZG_1_1KV,PNG_FOWLRTON_1KV,exact_bus,1.0,1.0,28.358900,-98.795097,28.371995,...,1.000000,0.641000,0.232500,0.331667,NaN,matched_low,False,False,True,one-to-one station-kV group; representative ra...
1,1000,FAYETT_1KV_1,DZG_1000_1KV,PNG_FAYETT_1KV,exact_bus,1.0,1.0,29.961300,-96.688400,29.979554,...,1.000000,0.670000,0.226500,0.396250,NaN,matched_low,True,False,True,one-to-one station-kV group; representative ra...
2,111138,BAC_1KV_1,DZG_111138_1KV,PNG_BAC_1KV,unmatched,1.0,1.0,29.505131,-94.981339,29.517361,...,0.500000,0.489286,0.062500,0.488333,NaN,unmatched,True,True,True,no reliable group-level candidate; weak statio...
3,111139_STAR,BAC_1KV_1,DZG_111139_STAR_1KV,PNG_BAC_1KV,ambiguous_within_group,1.0,1.0,29.505131,-94.981339,29.517361,...,0.500000,0.515000,0.021029,0.445000,NaN,ambiguous,True,True,True,group-level match is plausible but exact raw b...
4,111140_STAR,BAC_1KV_1,DZG_111140_STAR_1KV,PNG_BAC_1KV,ambiguous_within_group,1.0,1.0,29.505131,-94.981339,29.517361,...,0.500000,0.515000,0.021029,0.445000,NaN,ambiguous,True,True,True,group-level match is plausible but exact raw b...
5,112420,RAB_1KV_1,DZG_112420_1KV,PNG_RAB_1KV,unmatched,1.0,1.0,29.471901,-95.624298,29.479745,...,0.437500,0.466333,0.011833,0.523333,NaN,unmatched,True,True,True,no reliable group-level candidate; weak statio...
6,112421,RAB_1KV_1,DZG_112421_1KV,PNG_RAB_1KV,unmatched,1.0,1.0,29.471600,-95.624802,29.479745,...,0.437500,0.466333,0.011833,0.523333,NaN,unmatched,True,True,True,no reliable group-level candidate; weak statio...
7,112422,RAB_1KV_1,DZG_112422_1KV,PNG_RAB_1KV,unmatched,1.0,1.0,29.471001,-95.624298,29.479745,...,0.437500,0.466333,0.011833,0.523333,NaN,unmatched,True,True,True,no reliable group-level candidate; weak statio...
8,112423,RAB_1KV_1,DZG_112423_1KV,PNG_RAB_1KV,unmatched,1.0,1.0,29.470301,-95.624397,29.479745,...,0.437500,0.466333,0.011833,0.523333,NaN,unmatched,True,True,True,no reliable group-level candidate; weak statio...
9,112880,AZURE_1KV_1,DZG_112880_1KV,PNG_AZURE_1KV,unmatched,1.0,1.0,29.183001,-95.992996,32.998979,...,0.750000,0.455735,0.004235,0.170441,NaN,unmatched,True,False,True,no reliable group-level candidate; weak statio...


## Step 18 — Final interpretation notes

This notebook intentionally avoids claiming that every Dayzer bus has a uniquely correct raw Panorama bus.

The most important idea is:

> When two network models have different topology-processing or station-bus splitting conventions, the correct mapping may be group-level rather than exact raw-bus level.

This notebook keeps a broad candidate set. The purpose is to reduce false negatives during candidate generation, even if the notebook runs more slowly. The purpose is to reduce false negatives during candidate generation, even if the notebook runs more slowly.

Therefore, the final CSV contains both a proposed `pano_bus` and several review-oriented columns:

- `mapping_level`
- `confidence`
- `many_dayzer_groups_to_one_pano_group`
- `multi_member_group_flag`
- `needs_review`
- `reason`

For actual production use, the next improvements would be official object IDs, breaker/switch status, branch/equipment-level matching, and power-flow or market-behavior validation.
